# PlastiVision-Edge — reproducibility notebook

Code accompanying *PlastiVision-Edge: A Zero-Shot Open-Vocabulary Framework for Real-Time Plastic Waste Detection and Instance Segmentation on Edge Devices* (Balasundaram & Perumal, *Sustainability*, under review).

**What this notebook does**

| Part | Content | Paper item |
|---|---|---|
| 1 | Setup, configuration, models | – |
| 2 | PWVO ontology (47 categories; 34 benchmark-covered, 13 benchmark-uncovered, "BU") | Table 1, Fig. 2 |
| 3 | Zero-shot detection on the 13 BU categories, per-image isolated prompt | Fig. 7 |
| 4 | Replicate + negative-control evaluation, Wilson 95 % CIs | Table 10 |
| 5 | Inference modes: text prompt, visual prompt (SAVPE), prompt-free (LRPC) | Fig. 3 |
| 6 | TACO benchmark: category mapping, subset construction, evaluation, mIoU | Table 9 |
| 7 | TrashNet plastic-class detection rate | Table 9 |
| 8 | Prompt-design ablation | Table 5 |
| 9 | YOLO-World baseline | Table 9 |
| 10 | Plastic-coverage estimation | Fig. 4 |
| 11 | Edge export (ONNX / TFLite FP32 / INT8) and GPU latency | Tables 6–7 |
| A | Generators for the qualitative figures that use the authors' own photographs | Figs. 3, 4, 6 |

**How to run.** Run top to bottom on a GPU runtime (developed on a Colab T4). Each part sets its own vocabulary on the model, so parts can be re-run on their own after Part 1 and Part 2. Data locations are set in the *Configuration* cell. Nothing is downloaded or uploaded interactively except the datasets named in Parts 6 and 7.

**What is not in this repository.** The authors' photographs (zero-shot replicates and negative controls) are described in `data/README.md`; the TACO and TrashNet images are public datasets that are fetched by this notebook. Numbers printed by this notebook are the source of truth; the paper's values are listed in `README.md` for comparison and are not hard-coded anywhere below.

**Notes for readers.** TACO images are fetched from Flickr links that have partly rotted, so the evaluation subset can differ between runs. Part 6 therefore writes `results/taco_subset_manifest.csv` listing exactly the images used, and can rebuild from that manifest.

## Part 1 — Setup

In [ ]:
# Versions used for the reported results: Python 3.13, PyTorch 2.11 (CUDA 13.0), Ultralytics 8.4.177, NVIDIA T4.
# On Colab, torch/torchvision/numpy/pandas/matplotlib are preinstalled; only the extras are installed here.
# YOLOE's text encoder (MobileCLIP) is fetched by Ultralytics on first use and needs internet access.
%pip install -q "ultralytics==8.4.177" opencv-python-headless pycocotools seaborn tqdm requests

In [ ]:
import os, sys, json, time, shutil, math, glob, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import requests
import urllib.request
import torch
from PIL import Image
from tqdm.auto import tqdm
import ultralytics
from ultralytics import YOLO

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 110})
sns.set_style('whitegrid')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {ultralytics.__version__} | device: {DEVICE}')
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU found: everything runs, but the latency numbers are not comparable to the paper.')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Everything is relative to ROOT, so the notebook runs unchanged from a clone of the repository.
# Override with environment variables if the data live elsewhere (e.g. a mounted Drive).
ROOT       = Path(os.environ.get('PLASTIVISION_ROOT', '.')).resolve()
DATA_DIR   = ROOT / 'data'
RES_DIR    = ROOT / 'results'
EXP_DIR    = ROOT / 'exports'
ZS_DIR     = DATA_DIR / 'zeroshot_images'      # replicate photos of the 13 BU categories, named {pwvo_key}_{n}.jpg
NEG_DIR    = DATA_DIR / 'negative_controls'    # the 11 non-target control photographs (any file names)
SMP_DIR    = DATA_DIR / 'samples'              # any scene photographs for the demo cells in Part 5
TACO_DIR   = DATA_DIR / 'taco'
for d in (DATA_DIR, RES_DIR, EXP_DIR, ZS_DIR, NEG_DIR, SMP_DIR, TACO_DIR):
    d.mkdir(parents=True, exist_ok=True)

IMG_EXTS = ('.jpg', '.jpeg', '.png')

def list_images(folder):
    """All images under `folder` (recursive, case-insensitive extension match)."""
    return sorted(p for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)

print('ROOT =', ROOT)

## Part 2 — PWVO ontology

47 categories in six material super-classes. Each carries a text prompt for the open-vocabulary model and a flag `covered`: whether the category has a counterpart in the public benchmarks used here (TACO, TrashNet, ZeroWaste). The 13 categories with `covered=False` are the *benchmark-uncovered* (BU) set; they are evaluated by the zero-shot protocol in Parts 3–4, not by the detection benchmarks.

In [ ]:
PWVO = {
    # PET / Polyester (8)
    'pet_bottle':        {'prompt':'a plastic drinking bottle with cylindrical shape and screw cap, transparent or semi-transparent body, may have a coloured printed label, commonly used for water or soft drinks',        'super':'PET/Polyester',        'covered':True},
    'pet_tray':          {'prompt':'clear or lightly tinted rigid PET plastic tray, flat rectangular or oval base with a shallow raised rim, smooth hard surface, used for food packaging',        'super':'PET/Polyester',        'covered':False},
    'pet_clamshell':     {'prompt':'hinged transparent plastic clamshell container for fruit',              'super':'PET/Polyester',        'covered':True},
    'pet_film':          {'prompt':'thin crinkled transparent polyester plastic film sheet',                'super':'PET/Polyester',        'covered':True},
    'pet_blister_tray':  {'prompt':'formed rigid transparent plastic blister tray packaging',              'super':'PET/Polyester',        'covered':True},
    'polyester_fabric':  {'prompt':'synthetic polyester fabric scrap shiny and woven',                     'super':'PET/Polyester',        'covered':True},
    'pet_strap':         {'prompt':'flat transparent or blue plastic packing strap banding',               'super':'PET/Polyester',        'covered':True},
    'pet_bottle_cap':    {'prompt':'small plastic bottle cap round coloured on ground',                    'super':'PET/Polyester',        'covered':True},
    # HDPE / LDPE film (11)
    'carrier_bag':       {'prompt':'crumpled transparent or white plastic carrier bag discarded',          'super':'HDPE/LDPE film',       'covered':True},
    'cling_film':        {'prompt':'thin transparent plastic cling wrap food wrapping film',               'super':'HDPE/LDPE film',       'covered':False},
    'bubble_wrap':       {'prompt':'clear plastic sheet covered in a regular grid of small raised circular air-filled bubbles, cushioning padding texture, distinct bubble pattern across the entire surface',                 'super':'HDPE/LDPE film',       'covered':False},
    'shrink_wrap':       {'prompt':'thin clear plastic film heat-sealed tightly around the exact contour of a cylindrical or boxed object, taut and smooth with no looseness, visible seam line',               'super':'HDPE/LDPE film',       'covered':False},
    'hdpe_bottle':       {'prompt':'an opaque white or coloured plastic bottle with thick walls, used for milk, detergent or shampoo, cylindrical or oval shape with screw cap',      'super':'HDPE/LDPE film',       'covered':True},
    'plastic_bag':       {'prompt':'coloured thin plastic shopping bag crumpled outdoors',                 'super':'HDPE/LDPE film',       'covered':True},
    'bread_bag':         {'prompt':'clear plastic bag used for bread packaging tied at top',               'super':'HDPE/LDPE film',       'covered':True},
    'ldpe_film_sheet':   {'prompt':'large flat thin polyethylene plastic film sheet on ground',            'super':'HDPE/LDPE film',       'covered':True},
    'ziplock_bag':       {'prompt':'small resealable transparent zip-lock plastic bag',                    'super':'HDPE/LDPE film',       'covered':True},
    'produce_bag':       {'prompt':'thin perforated transparent plastic produce bag from supermarket',     'super':'HDPE/LDPE film',       'covered':True},
    'trash_bag':         {'prompt':'large black or dark green plastic bin liner trash bag',                'super':'HDPE/LDPE film',       'covered':True},
    # Polypropylene (9)
    'pp_food_container': {'prompt':'opaque white or coloured rigid polypropylene food container with lid', 'super':'Polypropylene',        'covered':True},
    'yoghurt_pot':       {'prompt':'small opaque plastic yoghurt pot with foil lid',                      'super':'Polypropylene',        'covered':True},
    'woven_pp_sack':     {'prompt':'woven white polypropylene rice or grain sack',                        'super':'Polypropylene',        'covered':False,
                         'prompt_long':'opaque white sack made of thick woven crosshatch plastic strands, coarse fabric-like texture, stiff structured sack shape with flat bottom, not a thin flimsy bag'},
    'pp_bottle_cap':     {'prompt':'coloured polypropylene bottle cap or flip-top closure',               'super':'Polypropylene',        'covered':True},
    'straw':             {'prompt':'a thin elongated plastic drinking straw, cylindrical and hollow, coloured or transparent, typically found near drinks or discarded on ground',                  'super':'Polypropylene',        'covered':True},
    'pp_cup':            {'prompt':'rigid opaque polypropylene disposable cup for hot drinks',             'super':'Polypropylene',        'covered':True},
    'cutlery':           {'prompt':'disposable plastic fork knife or spoon white or coloured',             'super':'Polypropylene',        'covered':True},
    'pp_tub':            {'prompt':'wide-mouth opaque polypropylene tub for margarine or cream',           'super':'Polypropylene',        'covered':True},
    'pp_woven_bag':      {'prompt':'flat woven polypropylene bag for cement or fertiliser',                'super':'Polypropylene',        'covered':True},
    # Polystyrene / EPS (7)
    'eps_foam_cup':      {'prompt':'a white expanded polystyrene foam disposable cup with grainy white texture, cylindrical shape, used for hot drinks or instant noodles',                      'super':'Polystyrene/EPS',      'covered':True},
    'eps_takeaway_box':  {'prompt':'a white expanded polystyrene foam takeaway food container or clamshell box with grainy white texture, rectangular shape with flat or hinged lid',              'super':'Polystyrene/EPS',      'covered':True},
    'eps_block':         {'prompt':'large block of white expanded polystyrene foam packaging',             'super':'Polystyrene/EPS',      'covered':False},
    'eps_tray':          {'prompt':'white polystyrene foam supermarket meat or vegetable tray',            'super':'Polystyrene/EPS',      'covered':True},
    'eps_fragment':      {'prompt':'broken white polystyrene foam pieces scattered on ground',             'super':'Polystyrene/EPS',      'covered':True},
    'eps_sheet':         {'prompt':'flat sheet of white polystyrene foam insulation board',                'super':'Polystyrene/EPS',      'covered':True},
    'ps_plastic_cup':    {'prompt':'clear rigid polystyrene transparent disposable plastic cup',           'super':'Polystyrene/EPS',      'covered':True},
    # Multilayer / composite (6)
    'chip_packet':       {'prompt':'shiny metallic-laminate chip or crisp snack packet discarded',        'super':'Multilayer/composite', 'covered':True},
    'retort_pouch':      {'prompt':'highly reflective gold or silver metallic foil-laminate pouch, mirror-like shiny surface, flexible with heat-sealed edges, noticeably more reflective than plain plastic film',          'super':'Multilayer/composite', 'covered':False},
    'blister_pack':      {'prompt':'pharmaceutical blister pack with aluminium foil and plastic bubbles', 'super':'Multilayer/composite', 'covered':False},
    'tetra_pak':         {'prompt':'rectangular Tetra Pak cardboard and plastic juice carton',            'super':'Multilayer/composite', 'covered':True},
    'multilayer_film':   {'prompt':'crumpled multilayer flexible plastic film from food packaging',       'super':'Multilayer/composite', 'covered':False,
                         'prompt_long':'crumpled thin flexible plastic film with multiple bonded layers visible as a laminate, loosely wrinkled sheet from food packaging, distinct from a rigid woven sack, a standing sealed pouch, or a coiled rope'},
    'coffee_pod':        {'prompt':'small single-use plastic and aluminium coffee pod capsule',           'super':'Multilayer/composite', 'covered':True},
    # Other (6)
    'microplastic_frag': {'prompt':'small coloured plastic fragment chip or pellet on ground',            'super':'Other',               'covered':False},
    'rubber_band':       {'prompt':'small thin circular elastic rubber loop, brightly coloured (red, yellow, green, orange), stretchable band, often several bunched together in a small pile',                          'super':'Other',               'covered':False},
    'synthetic_rope':    {'prompt':'thick multi-strand twisted or braided plastic cord, coiled in loops, individual fibres may fray at the cut ends, white grey or blue colour, rope-like texture distinct from flat film or fabric',                'super':'Other',               'covered':False},
    'pvc_pipe':          {'prompt':'rigid hollow cylindrical tube with hard smooth uniform-diameter walls, straight pipe shape, may have a socket joint at one end, distinctly rigid unlike flexible film or sheet',                   'super':'Other',               'covered':False},
    'cable_tie':         {'prompt':'small black or white plastic zip tie cable fastener',                 'super':'Other',               'covered':True},
    'plastic_sticker':   {'prompt':'peeled plastic label or sticker sheet discarded',                    'super':'Other',               'covered':True},
}
# Longer descriptive versions of the remaining prompts. The reported runs use 'prompt'; 'prompt_long' is released for reference and was not used in the reported runs.
prompt_long_extra = {
    "pet_clamshell": "hinged transparent rigid plastic clamshell container with a snap-close lid, glossy clear walls, shallow rectangular shape, used for packaging fruit, berries or salad",
    "pet_film": "thin crinkled transparent polyester plastic film sheet, glossy and slightly stiff, crumpled or folded with sharp creases, discarded flat or loosely balled",
    "pet_blister_tray": "formed rigid transparent plastic blister tray with moulded compartments and a flat rim, glossy clear surface, used to hold and display packaged products",
    "polyester_fabric": "scrap of synthetic polyester fabric, shiny woven or knitted textile surface with a visible weave, soft flexible cloth offcut, often creased or frayed at the edges",
    "pet_strap": "flat narrow plastic packing strap or banding, smooth glossy surface, transparent or blue or green colour, long ribbon-like shape often coiled or cut into lengths",
    "pet_bottle_cap": "small round plastic bottle cap with a ridged edge and screw thread inside, brightly coloured, lying on the ground, much smaller than the bottle it came from",
    "carrier_bag": "crumpled thin transparent or white plastic carrier bag with two loop handles, loosely wrinkled and translucent, discarded on the ground or caught on vegetation",
    "cling_film": "very thin transparent plastic cling wrap clinging to itself in wrinkled folds, glossy clear film with no fixed shape, used for wrapping food, often balled or stretched",
    "plastic_bag": "coloured thin plastic shopping bag, crumpled and wrinkled outdoors, soft flexible film with handles or a knotted top, often printed with shop branding",
    "bread_bag": "clear thin plastic bag used for packaging bread, loosely filled or empty, tied or clipped at the top with a plastic tag, transparent film with crinkles",
    "ldpe_film_sheet": "large flat thin polyethylene plastic film sheet lying on the ground, smooth translucent or opaque sheet with wide folds, used for covering, wrapping or agricultural mulch",
    "ziplock_bag": "small transparent resealable zip-lock plastic bag with a visible interlocking grip strip along the top, clear flexible film, flat or slightly wrinkled",
    "produce_bag": "thin transparent plastic produce bag from a supermarket, lightweight and clear, rolled or crumpled, with small perforations or a twist-tie top",
    "trash_bag": "large black or dark green plastic bin liner trash bag, thick glossy opaque film, knotted at the top, often bulging when filled with waste",
    "pp_food_container": "opaque white or coloured rigid polypropylene food container with a snap-on lid, smooth matte walls, rectangular or round tub shape, used for takeaway or storage",
    "yoghurt_pot": "small opaque plastic yoghurt pot, tapered cup shape with a thin rim, white or coloured, often with a torn foil lid and a printed label",
    "pp_bottle_cap": "coloured polypropylene bottle cap or flip-top closure, small round hard plastic lid with a hinged or threaded design, found separately from its bottle",
    "pp_cup": "rigid opaque polypropylene disposable cup for hot drinks, tapered cylindrical shape with a rolled rim, smooth matte walls, white or coloured, often with a printed pattern",
    "cutlery": "disposable plastic fork, knife or spoon, rigid thin handle with a moulded head, white or coloured, lying singly or in small groups on the ground",
    "pp_tub": "wide-mouth opaque polypropylene tub for margarine or cream, short round container with a thick rim and a snap-on lid, white or coloured with a printed label",
    "pp_woven_bag": "flat woven polypropylene bag for cement or fertiliser, coarse crosshatch strands, stiff rectangular sack shape, often printed with large lettering, white or coloured",
    "eps_block": "large block of white expanded polystyrene foam packaging, grainy bead-like texture, lightweight rectangular shape with moulded cavities and crumbling edges",
    "eps_tray": "white polystyrene foam supermarket tray for meat or vegetables, shallow rectangular shape with a raised rim, grainy matte surface, often stained",
    "eps_fragment": "broken white polystyrene foam pieces of irregular size scattered on the ground, grainy bead-like texture, lightweight crumbly chunks with rough broken edges",
    "eps_sheet": "flat sheet of white polystyrene foam insulation board, smooth grainy surface, uniform thickness with straight or snapped edges, rigid but lightweight",
    "ps_plastic_cup": "clear rigid polystyrene disposable plastic cup, transparent glossy walls, tapered cylindrical shape with a thin rim, used for cold drinks, brittle and often cracked",
    "chip_packet": "shiny metallic-laminate chip or crisp snack packet, crinkled foil-like surface with bright printed graphics, heat-sealed edges, flat or puffed, discarded",
    "blister_pack": "pharmaceutical blister pack with a sheet of aluminium foil on one side and rows of transparent plastic bubbles holding tablets, flat rectangular strip",
    "tetra_pak": "rectangular Tetra Pak carton for juice or milk, rigid cardboard-like body with plastic and aluminium lining, printed graphics, folded flat top and a screw cap or straw hole",
    "coffee_pod": "small single-use coffee pod capsule, short cylindrical plastic cup with an aluminium foil lid and a ridged rim, often dark coloured and stained",
    "microplastic_frag": "small coloured plastic fragment, chip or pellet a few millimetres across, hard irregular or rounded shape, bright colour against soil, sand or pavement",
    "cable_tie": "small black or white plastic zip tie cable fastener, thin ribbed strap with a square locking head, often looped closed or cut open, lying on the ground",
    "plastic_sticker": "peeled plastic label or sticker sheet, thin glossy adhesive-backed film with printed graphics, curled or creased edges, discarded or stuck to a surface"
}
for _k, _v in PWVO.items():
    _v.setdefault('prompt_long', prompt_long_extra.get(_k, _v['prompt']))

COVERED_CATS = [k for k, v in PWVO.items() if v['covered']]
BU_CATS      = [k for k, v in PWVO.items() if not v['covered']]
ALL_KEYS     = list(PWVO)
ALL_PROMPTS  = [PWVO[k]['prompt'] for k in ALL_KEYS]

SUPER_COLORS = {
    'PET/Polyester':        '#1D9E75',
    'HDPE/LDPE film':       '#378ADD',
    'Polypropylene':        '#D85A30',
    'Polystyrene/EPS':      '#BA7517',
    'Multilayer/composite': '#D4537E',
    'Other':                '#888780',
}

assert len(PWVO) == 47, len(PWVO)
assert len(BU_CATS) == 13 and len(COVERED_CATS) == 34, (len(BU_CATS), len(COVERED_CATS))
with open(RES_DIR / 'pwvo.json', 'w') as f:
    json.dump(PWVO, f, indent=2)
print(f'PWVO: {len(PWVO)} categories | {len(COVERED_CATS)} benchmark-covered | {len(BU_CATS)} benchmark-uncovered (BU)')
print('BU:', BU_CATS)


In [ ]:
# Fig. 2 — categories per material, and covered / uncovered split
df_p = pd.DataFrame([{'Category': k, 'Super': v['super'], 'Split': 'covered' if v['covered'] else 'BU'} for k, v in PWVO.items()])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sc = df_p.groupby('Super')['Category'].count().sort_values()
axes[0].barh(sc.index, sc.values, color='#1D9E75', edgecolor='white', height=0.6)
for i, v in enumerate(sc.values):
    axes[0].text(v + .1, i, str(v), va='center', fontsize=10)
axes[0].set_xlabel('Categories'); axes[0].set_title('PWVO — categories per material', fontweight='bold')
axes[1].pie([len(COVERED_CATS), len(BU_CATS)],
            labels=[f'Benchmark-covered ({len(COVERED_CATS)})', f'Benchmark-uncovered ({len(BU_CATS)})'],
            colors=['#1D9E75', '#E1F5EE'], autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2}, textprops={'fontsize': 12})
axes[1].set_title('Benchmark coverage of the 47 categories', fontweight='bold')
plt.tight_layout(); plt.savefig(RES_DIR / 'pwvo.png', dpi=140, bbox_inches='tight'); plt.show()
print(df_p.groupby(['Super', 'Split'])['Category'].count().unstack(fill_value=0).to_string())

## Part 1b — Models and shared helpers

Three YOLOE-26 checkpoints (Ultralytics): **S** is the edge model, **L** the accuracy reference, **S-pf** the prompt-free variant. `set_vocab` is the only place where a model's vocabulary is changed; it returns the list of keys in class-index order, so later code never has to assume that class *i* means PWVO key *i*.

In [ ]:
def load_yoloe(weights):
    m = YOLO(weights)
    m.model.to(DEVICE)
    return m

model_s  = load_yoloe('yoloe-26s-seg.pt')
model_l  = load_yoloe('yoloe-26l-seg.pt')
model_pf = load_yoloe('yoloe-26s-seg-pf.pt')

def n_params_m(m):
    return sum(p.numel() for p in m.model.parameters()) / 1e6

for name, m in [('YOLOE-26S', model_s), ('YOLOE-26L', model_l), ('YOLOE-26S-pf', model_pf)]:
    print(f'{name:14s} task={m.task:8s} params={n_params_m(m):5.1f} M (unfused)')

In [ ]:
def force_model_to_device(model, device=DEVICE):
    """set_classes() can leave text-embedding buffers on the CPU; push everything back."""
    try:
        model.model.to(device)
        for module in model.model.modules():
            for name, buf in module.named_buffers(recurse=False):
                try: setattr(module, name, buf.to(device))
                except Exception: pass
    except Exception:
        pass

def set_vocab(model, keys):
    """Set `model`'s open vocabulary to the PWVO prompts of `keys` (class i == keys[i]). Returns `keys`."""
    keys = list(keys)
    prompts = [PWVO[k]['prompt'] for k in keys]
    model.set_classes(prompts, model.get_text_pe(prompts))
    force_model_to_device(model)
    return keys

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))

def draw_detections(img_path, result, labels, conf_thr=0.0, mask_alpha=0.42, tag_fn=None):
    """Return (RGB image with masks, boxes and labels drawn, list of detection dicts).
    `labels[i]` is the PWVO key of model class i."""
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    canvas = img.astype(np.float32)
    log = []
    r = result
    if r.boxes is None or len(r.boxes) == 0:
        return img, log
    masks = r.masks.data.cpu().numpy() if r.masks is not None else None
    # Blend all masks first, then draw outlines/boxes/labels on the final uint8 image
    # (the earlier version drew on temporary copies, so outlines, boxes and labels were lost).
    items = []
    for i, box in enumerate(r.boxes):
        conf = float(box.conf)
        if conf < conf_thr:
            continue
        key = labels[int(box.cls)] if int(box.cls) < len(labels) else f'cls_{int(box.cls)}'
        sup = PWVO.get(key, {}).get('super', 'Other')
        col = hex_to_rgb(SUPER_COLORS.get(sup, '#888780'))
        binary = None
        if masks is not None and i < len(masks):
            binary = (cv2.resize(masks[i], (W, H), interpolation=cv2.INTER_LINEAR) > 0.5)
            for ch in range(3):
                canvas[..., ch] = np.where(binary, canvas[..., ch] * (1 - mask_alpha) + col[ch] * mask_alpha, canvas[..., ch])
        xyxy = [int(v) for v in box.xyxy[0]]
        items.append((key, conf, col, binary, xyxy))
        log.append({'category': key, 'super': sup, 'confidence': round(conf, 3), 'bbox': xyxy,
                    'mask_px': int(binary.sum()) if binary is not None else 0,
                    'mask_pct_of_frame': round(float(binary.mean()) * 100, 3) if binary is not None else 0.0})
    out = np.clip(canvas, 0, 255).astype(np.uint8)
    fs = max(0.6, min(1.2, W / 2000)); th_txt = max(1, int(fs * 2))
    for key, conf, col, binary, (x1, y1, x2, y2) in items:
        if binary is not None:
            cnts, _ = cv2.findContours(binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(out, cnts, -1, col, 3)
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 3)
        tag = tag_fn(key, conf) if tag_fn else f"{key.replace('_', ' ')}  {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, fs, th_txt)
        pad = int(th * 0.4); yb = max(y1 - th - 2 * pad, 0)
        cv2.rectangle(out, (x1, yb), (x1 + tw + 2 * pad, y1), col, -1)
        cv2.putText(out, tag, (x1 + pad, y1 - pad), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), th_txt)
    return out, log

def show_detection(img_path, result, labels, conf_thr=0.0, title='', save_as=None):
    vis, log = draw_detections(img_path, result, labels, conf_thr)
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)); axes[0].set_title('Input'); axes[0].axis('off')
    axes[1].imshow(vis); axes[1].set_title(title or f'{len(log)} detection(s)'); axes[1].axis('off')
    plt.tight_layout()
    if save_as: plt.savefig(RES_DIR / save_as, dpi=130, bbox_inches='tight')
    plt.show()
    return log

def wilson_ci(successes, n, z=1.96):
    """Wilson score interval in percent: (point, low, high)."""
    if n == 0: return (0.0, 0.0, 0.0)
    p = successes / n
    denom = 1 + z ** 2 / n
    centre = p + z ** 2 / (2 * n)
    adj = z * math.sqrt((p * (1 - p) + z ** 2 / (4 * n)) / n)
    return (p * 100, max(0.0, (centre - adj) / denom) * 100, min(1.0, (centre + adj) / denom) * 100)

hm = lambda a, b: 2 * a * b / (a + b + 1e-9)    # harmonic mean
print('helpers ready')

## Part 3 — Zero-shot detection on the 13 BU categories (Fig. 7)

**Protocol.** For every image the vocabulary is set to the *single* PWVO prompt of the image's own category (`set_classes([prompt])`), so no other category competes. Confidence threshold 0.10, YOLOE-26S. This is the per-image isolated protocol used for Fig. 7 and Table 10; evaluating with all 47 prompts active is a different (harder) protocol and gives different results.

**Input.** Photographs in `data/zeroshot_images/` whose file names start with the PWVO key (e.g. `eps_block_2.jpeg`). The first match per category is used for the figure.

In [ ]:
ZS_CONF = 0.10

files_zs = list_images(ZS_DIR)
available = {}
for k in BU_CATS:
    hits = [p for p in files_zs if p.name.lower().startswith(k.lower())]
    if hits: available[k] = hits[0]
print(f'{len(files_zs)} image(s) in {ZS_DIR.name}/ | matched {len(available)}/{len(BU_CATS)} BU categories')
missing = [k for k in BU_CATS if k not in available]
if missing: print('no image for:', missing)

zs_results = {}
for key, path in available.items():
    set_vocab(model_s, [key])                      # one prompt, class 0 == key
    r = model_s.predict(str(path), conf=ZS_CONF, device=DEVICE, verbose=False)[0]
    best, box = 0.0, None
    if r.boxes is not None and len(r.boxes):
        j = int(r.boxes.conf.argmax()); best = float(r.boxes.conf[j]); box = r.boxes.xyxyn[j].tolist()
    zs_results[key] = dict(conf=best, box=box, detected=best >= ZS_CONF, img=path)

n = len(zs_results)
n_det = sum(v['detected'] for v in zs_results.values())
n_strong = sum(v['conf'] >= 0.50 for v in zs_results.values())
print(f'\nDetected (conf >= {ZS_CONF}): {n_det}/{n} | strong (conf >= 0.50): {n_strong}/{n}')
for k, v in sorted(zs_results.items(), key=lambda kv: -kv[1]['conf']):
    print(f"  {'detected' if v['detected'] else 'missed  '}  {k:<20} conf={v['conf']:.3f}  [{PWVO[k]['super']}]")
pd.DataFrame([{'category': k, 'image': v['img'].name, 'best_conf': round(v['conf'], 4), 'detected': v['detected']}
              for k, v in zs_results.items()]).to_csv(RES_DIR / 'zeroshot_per_image.csv', index=False)

In [ ]:
# Fig. 7 — one panel per BU category; every panel is tagged [BU]
if zs_results:
    ncols = 4; nrows = math.ceil(len(zs_results) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
    axes = np.atleast_1d(axes).flatten()
    for ax, (key, res) in zip(axes, zs_results.items()):
        img = np.array(Image.open(res['img']).convert('RGB')); H, W = img.shape[:2]
        ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        color = SUPER_COLORS[PWVO[key]['super']]
        if res['box']:
            x1, y1, x2, y2 = res['box']
            ax.add_patch(mpatches.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H, linewidth=2.5, edgecolor=color, facecolor='none'))
            ax.text(x1 * W, max(y1 * H - 4, 12), f"{'✓' if res['detected'] else '✗'} {key} {res['conf']:.2f}", color='white',
                    fontsize=7, fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', fc=color, alpha=0.85))
        ax.set_title(f'{key}\n[BU]', fontsize=8, color='green' if res['detected'] else 'red', pad=3)
        for sp in ax.spines.values():
            sp.set_edgecolor(color if res['detected'] else '#cccccc'); sp.set_linewidth(2.5 if res['detected'] else 1.0)
    for ax in axes[len(zs_results):]: ax.set_visible(False)
    fig.suptitle('PlastiVision-Edge — zero-shot detection on the benchmark-uncovered PWVO categories\n'
                 'YOLOE-26S + per-image PWVO prompt | ✓ detected | ✗ not detected', fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.savefig(RES_DIR / 'zeroshot_33_categories.png', dpi=120, bbox_inches='tight', facecolor='white'); plt.show()
else:
    print('No images found — see data/README.md for the expected file names.')

## Part 4 — Replicate and negative-control evaluation (Table 10)

Same isolated-prompt protocol as Part 3, confidence ≥ 0.10, but over **all** replicate photographs of each BU category (`{pwvo_key}_{n}.jpg`) plus a shared pool of **non-target control** photographs in `data/negative_controls/`. Each control photograph is tested against every category's prompt; a detection at conf ≥ 0.10 counts as a false positive. The controls are everyday objects that resemble a BU category in shape, colour or texture without being an instance of any of the 13 (two of them, a clear acrylic ruler and white paper in a clear polybag, contain incidental plastic, which is why they are called *non-target* rather than *non-plastic*). Success and false-positive rates are reported with Wilson score 95 % intervals, per category and pooled.

In [ ]:
TAB10_CONF = 0.10

def best_conf(model, img_path, conf=TAB10_CONF):
    r = model.predict(str(img_path), conf=conf, device=DEVICE, verbose=False)[0]
    return float(r.boxes.conf.max()) if (r.boxes is not None and len(r.boxes)) else 0.0

pos_files = list_images(ZS_DIR)
neg_files = list_images(NEG_DIR)
print(f'{len(neg_files)} non-target control image(s):')
for p in neg_files: print('  -', p.name)

tab10, absent = {}, []
for cat in BU_CATS:
    reps = [p for p in pos_files if p.name.lower().startswith(cat.lower())]
    if not reps:
        absent.append(cat); continue
    set_vocab(model_s, [cat])
    confs = [best_conf(model_s, p) for p in reps]
    succ = sum(c >= TAB10_CONF for c in confs)
    fp = sum(best_conf(model_s, p) >= TAB10_CONF for p in neg_files)
    s_rate, s_lo, s_hi = wilson_ci(succ, len(reps))
    f_rate, f_lo, f_hi = wilson_ci(fp, len(neg_files))
    tab10[cat] = dict(n=len(reps), successes=succ, success_rate=s_rate, s_lo=s_lo, s_hi=s_hi,
                      n_neg=len(neg_files), fp=fp, fp_rate=f_rate, f_lo=f_lo, f_hi=f_hi,
                      confs=[round(c, 3) for c in confs], replicates=[p.name for p in reps])
    print(f'{cat:<18} succ {succ}/{len(reps)} ({s_rate:5.1f}% [{s_lo:5.1f},{s_hi:5.1f}])   '
          f'FP {fp}/{len(neg_files)} ({f_rate:5.1f}% [{f_lo:5.1f},{f_hi:5.1f}])')

tot_n    = sum(v['n'] for v in tab10.values());        tot_s = sum(v['successes'] for v in tab10.values())
tot_negn = sum(v['n_neg'] for v in tab10.values());    tot_f = sum(v['fp'] for v in tab10.values())
p_s, p_slo, p_shi = wilson_ci(tot_s, tot_n); p_f, p_flo, p_fhi = wilson_ci(tot_f, tot_negn)
print(f'\nPOOLED success : {tot_s}/{tot_n} = {p_s:.1f}% [{p_slo:.1f}, {p_shi:.1f}]')
print(f'POOLED false+  : {tot_f}/{tot_negn} = {p_f:.1f}% [{p_flo:.1f}, {p_fhi:.1f}]  ({len(tab10)} categories x {len(neg_files)} controls)')
if absent: print('no replicate photographs for:', absent)

pd.DataFrame([{**{'category': k}, **{kk: vv for kk, vv in v.items() if kk not in ('confs', 'replicates')},
               'confs': ' '.join(map(str, v['confs'])), 'replicates': ' '.join(v['replicates'])} for k, v in tab10.items()]
             ).to_csv(RES_DIR / 'table10_replicates_negcontrols.csv', index=False)

## Part 5 — Inference modes (qualitative demo)

Put one or more scene photographs in `data/samples/`; the first one is used. The three modes are
(1) **text-prompted**: the PWVO prompts define the vocabulary; (2) **visual-prompted (SAVPE)**: a cropped example of the object is the prompt; (3) **prompt-free (LRPC)**: the model's built-in vocabulary, no prompts.

In [ ]:
sample_files = list_images(SMP_DIR)
HAVE_SAMPLE = bool(sample_files)
primary_path = sample_files[0] if HAVE_SAMPLE else None
print('Demo image:', primary_path.name if HAVE_SAMPLE else f'none — put a scene photograph in {SMP_DIR} to run Part 5 (the cells below are skipped)')

if HAVE_SAMPLE:
    # Mode 1 — text-prompted, a 10-category working vocabulary
    DEMO_CATS = ['pet_bottle', 'carrier_bag', 'chip_packet', 'eps_foam_cup', 'eps_takeaway_box',
                 'plastic_bag', 'hdpe_bottle', 'straw', 'tetra_pak', 'pp_food_container']
    demo_labels = set_vocab(model_s, DEMO_CATS)
    res1 = model_s.predict(str(primary_path), conf=0.20, device=DEVICE, verbose=False)[0]
    log1 = show_detection(primary_path, res1, demo_labels, title='Mode 1 — text-prompted (YOLOE-26S)', save_as='mode1_text_prompted.png')
    print(pd.DataFrame(log1).to_string(index=False) if log1 else 'no detections at conf >= 0.20')

In [ ]:
if HAVE_SAMPLE:
    # Confidence filtering: the same image under the full 47-category vocabulary (YOLOE-26L) at two thresholds.
    # Low thresholds admit visually similar non-plastic objects (e.g. cardboard read as polystyrene); the
    # deployment threshold used in the paper for outdoor scenes is 0.30.
    full_labels = set_vocab(model_l, ALL_KEYS)
    rows = []
    for thr in (0.15, 0.30):
        r = model_l.predict(str(primary_path), conf=thr, device=DEVICE, verbose=False)[0]
        img_area = cv2.imread(str(primary_path)).shape[0] * cv2.imread(str(primary_path)).shape[1]
        for b in (r.boxes or []):
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
            rows.append({'conf_thr': thr, 'category': full_labels[int(b.cls)], 'confidence': round(float(b.conf), 3),
                         'bbox_area_pct': round((x2 - x1) * (y2 - y1) / img_area * 100, 2)})
    df_fp = pd.DataFrame(rows)
    print(df_fp.to_string(index=False) if len(df_fp) else 'no detections')
    df_fp.to_csv(RES_DIR / f'confidence_sweep_{primary_path.stem}.csv', index=False)
    r30 = model_l.predict(str(primary_path), conf=0.30, device=DEVICE, verbose=False)[0]
    show_detection(primary_path, r30, full_labels, title='YOLOE-26L + full PWVO, conf >= 0.30', save_as='mode1_conf030.png');

In [ ]:
if HAVE_SAMPLE:
    # Mode 2 — visual-prompted (SAVPE). The reference is a crop of the best Mode-1 detection; the bounding box
    # covers the whole crop. (Using the full scene as its own reference collapses the similarity signal.)
    from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

    if len(res1.boxes) == 0:
        print('Mode 1 found nothing to crop; skipping Mode 2.')
    else:
        best = res1.boxes[res1.boxes.conf.argmax()]
        x1, y1, x2, y2 = [int(v) for v in best.xyxy[0]]
        crop = cv2.imread(str(primary_path))[y1:y2, x1:x2]
        ref_path = SMP_DIR / f'savpe_ref_{primary_path.stem}.jpg'
        cv2.imwrite(str(ref_path), crop)
        ch, cw = crop.shape[:2]
        vp = {'bboxes': np.array([[0, 0, cw, ch]], dtype=np.float32), 'cls': np.array([0], dtype=np.int32)}   # box = whole crop, in the reference crop's own pixel coordinates
        model_vp = load_yoloe('yoloe-26s-seg.pt')
        res2 = None
        for conf_try in (0.05, 0.03, 0.01):
            res2 = model_vp.predict(source=str(primary_path), refer_image=str(ref_path), visual_prompts=vp,
                                    predictor=YOLOEVPSegPredictor, conf=conf_try, device=DEVICE, verbose=False)[0]
            print(f'conf >= {conf_try}: {len(res2.boxes)} detection(s)')
            if len(res2.boxes): break
        if res2 is not None and len(res2.boxes):
            show_detection(primary_path, res2, ['visual match'], title='Mode 2 — visual-prompted (SAVPE)', save_as='mode2_savpe.png')

    # Mode 3 — prompt-free (LRPC): no set_classes(); the model's internal vocabulary is used
    res3 = model_pf.predict(str(primary_path), conf=0.25, device=DEVICE, verbose=False)[0]
    print(f'\nMode 3 — prompt-free: {len(res3.boxes)} box(es), internal vocabulary of {len(model_pf.names)} classes')
    for b in res3.boxes:
        print(f'  [{model_pf.names[int(b.cls)]}] conf={float(b.conf):.3f}')

## Part 6 — TACO benchmark (Table 9)

### 6.1 Category mapping
TACO has 60 categories; 18 of them map onto 13 PWVO categories (a 14th, `plastic_bag`, has no reachable TACO category and is excluded from the mean). The mapping is a judgement call: for example *Plastic film* → `pet_film` and *Other plastic wrapper* → `chip_packet` are semantic approximations, and these two classes carry most of the instances. The mapping is listed in full so a reader can change it and re-run.

In [ ]:
# TACO category name -> PWVO key. Only names that exist in TACO's 60-category list are included
# (check_taco_mapping() below verifies this against the downloaded annotations).
TACO_TO_PWVO = {
    'Clear plastic bottle':      'pet_bottle',
    'Other plastic bottle':      'pet_bottle',
    'Plastic bottle cap':        'pet_bottle_cap',
    'Single-use carrier bag':    'carrier_bag',
    'Plastic film':              'pet_film',
    'Foam cup':                  'eps_foam_cup',
    'Disposable food container': 'eps_takeaway_box',
    'Foam food container':       'eps_takeaway_box',
    'Crisp packet':              'chip_packet',
    'Other plastic wrapper':     'chip_packet',
    'Meal carton':               'tetra_pak',
    'Drink carton':              'tetra_pak',
    'Plastic straw':             'straw',
    'Plastic utensils':          'cutlery',
    'Garbage bag':               'trash_bag',
    'Aluminium blister pack':    'blister_pack',
    'Carded blister pack':       'blister_pack',
    'Squeezable tube':           'hdpe_bottle',
}
# PWVO classes in order of first appearance; this order defines the YOLO label ids written below.
# plastic_bag has no TACO source, so it is appended as the 14th (zero-instance) class to keep the
# 14-class layout of the reported evaluation.
TACO_BENCH_CATS = list(dict.fromkeys(TACO_TO_PWVO.values())) + ['plastic_bag']
TACO_IDX = {c: i for i, c in enumerate(TACO_BENCH_CATS)}
TACO_COVERED = [c for c in TACO_BENCH_CATS if PWVO[c]['covered']]
TACO_BU      = [c for c in TACO_BENCH_CATS if not PWVO[c]['covered']]
missing_keys = [c for c in TACO_BENCH_CATS if c not in PWVO]
assert not missing_keys, missing_keys
print(f'{len(TACO_TO_PWVO)} TACO categories -> {len(TACO_BENCH_CATS)} PWVO classes ({len(TACO_COVERED)} covered, {len(TACO_BU)} BU)')
print('BU classes in TACO:', TACO_BU)

In [ ]:
# ── 6.2 TACO annotations ──
TACO_ANN_URL  = 'https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json'
TACO_ANN_FILE = TACO_DIR / 'annotations.json'
if not TACO_ANN_FILE.exists():
    r = requests.get(TACO_ANN_URL, timeout=60); r.raise_for_status()
    TACO_ANN_FILE.write_bytes(r.content)
taco_ann = json.loads(TACO_ANN_FILE.read_text())
real_names = {c['name'] for c in taco_ann['categories']}
unknown = sorted(set(TACO_TO_PWVO) - real_names)
print(f"TACO: {len(taco_ann['images'])} images, {len(taco_ann['annotations'])} annotations, {len(real_names)} categories")
print('mapping keys that do not exist in TACO:', unknown or 'none')
assert not unknown, 'fix TACO_TO_PWVO'

### 6.3 Evaluation subset

Annotations with area < 300 px are dropped; images are downloaded from their Flickr URLs, and about half of those links no longer resolve, so the set of images that can be fetched changes over time. To make the evaluation reproducible the notebook works from a **manifest**: `results/taco_subset_manifest.csv` lists the image ids that were evaluated. If the manifest exists, only those images are fetched; otherwise it is built from whatever downloads succeed and then written. Commit the manifest with the results.

In [ ]:
MIN_MASK_AREA_PX = 300       # annotations smaller than this are ignored
TACO_DL_DELAY    = 0.15      # seconds between Flickr requests
MANIFEST         = RES_DIR / 'taco_subset_manifest.csv'
TACO_IMGS, TACO_LBLS = TACO_DIR / 'images', TACO_DIR / 'labels'
TACO_IMGS.mkdir(exist_ok=True); TACO_LBLS.mkdir(exist_ok=True)

def coco_poly_to_yolo(poly_flat, w, h):
    return ' '.join(f'{max(0., min(1., poly_flat[i] / w)):.5f} {max(0., min(1., poly_flat[i + 1] / h)):.5f}'
                    for i in range(0, len(poly_flat) - 1, 2))

cat_to_pwvo = {c['id']: TACO_TO_PWVO[c['name']] for c in taco_ann['categories'] if c['name'] in TACO_TO_PWVO}
img_lookup  = {im['id']: im for im in taco_ann['images']}
ann_by_img  = {}
for a in taco_ann['annotations']:
    if a['category_id'] in cat_to_pwvo and a.get('area', 0) >= MIN_MASK_AREA_PX:
        ann_by_img.setdefault(a['image_id'], []).append(a)
candidates = sorted(ann_by_img)                                    # deterministic order
if MANIFEST.exists():
    wanted = set(pd.read_csv(MANIFEST)['image_id'])
    candidates = [i for i in candidates if i in wanted]
    print(f'Using frozen manifest: {len(candidates)} image ids')
else:
    print(f'No manifest yet: trying {len(candidates)} candidate images')

converted, dl_failed, counts, manifest_rows = 0, 0, Counter(), []
for img_id in tqdm(candidates, desc='TACO'):
    info = img_lookup[img_id]; W, H = info['width'], info['height']
    url = info.get('flickr_url') or info.get('coco_url', '')
    ip, lp = TACO_IMGS / f'taco_{img_id:06d}.jpg', TACO_LBLS / f'taco_{img_id:06d}.txt'
    if not ip.exists():
        try:
            urllib.request.urlretrieve(url, str(ip)); time.sleep(TACO_DL_DELAY)
        except Exception:
            dl_failed += 1; continue
    if cv2.imread(str(ip)) is None:                 # truncated / corrupt download
        ip.unlink(missing_ok=True); dl_failed += 1; continue
    lines = []
    for a in ann_by_img[img_id]:
        segs = a.get('segmentation') or []
        poly = max(segs, key=len) if segs else []
        if len(poly) < 6: continue
        key = cat_to_pwvo[a['category_id']]
        lines.append(f'{TACO_IDX[key]} {coco_poly_to_yolo(poly, W, H)}'); counts[key] += 1
    if lines:
        lp.write_text('\n'.join(lines)); converted += 1
        manifest_rows.append({'image_id': img_id, 'file': ip.name, 'url': url})
    elif ip.exists():
        ip.unlink()

if not MANIFEST.exists():
    pd.DataFrame(manifest_rows).to_csv(MANIFEST, index=False)
(TACO_DIR / 'data.yaml').write_text(f'path: {TACO_DIR}\ntrain: images\nval: images\nnc: {len(TACO_BENCH_CATS)}\nnames: {TACO_BENCH_CATS}\n')

print(f'\nconverted {converted} images | download failures {dl_failed} | labels {sum(counts.values())}')
df_dist = pd.DataFrame([{'class': k, 'instances': counts.get(k, 0), 'split': 'covered' if PWVO[k]['covered'] else 'BU'}
                        for k in TACO_BENCH_CATS]).sort_values('instances', ascending=False)
print(df_dist.to_string(index=False)); df_dist.to_csv(RES_DIR / 'taco_class_distribution.csv', index=False)

### 6.4 Evaluator

One evaluator is used for every TACO number below (Table 9, the ablation, mIoU, the YOLO-World baseline), so they cannot drift apart.

* Protocol: confidence ≥ 0.20, IoU ≥ 0.5 (box and mask), PASCAL-VOC 11-point AP, greedy matching in descending confidence, each ground truth matched once.
* The model's class index is translated to a PWVO key through `model_keys` (the keys the model's vocabulary was set to), so the same evaluator is valid for the full 47-category vocabulary and for any subset vocabulary.
* **mAP is averaged over estimable classes only** (classes with ≥ 1 ground-truth instance). Averaging over all 14 classes, including `plastic_bag` with no instances, would pull the mean down by construction; the raw 14-class mean is also returned for comparison.
* Mask predictions are taken in the same confidence order as the boxes (the earlier version indexed the unsorted mask array with the sorted rank).
* Also returned: instance mIoU over matched pairs and an image-level semantic IoU.

In [ ]:
def parse_yolo_seg_label(lbl_path, img_path):
    h, w = cv2.imread(str(img_path)).shape[:2]     # unreadable images never enter the subset (see 6.3)
    objs = []
    p = Path(lbl_path)
    if not p.exists(): return objs
    for line in p.read_text().strip().split('\n'):
        parts = line.split()
        if len(parts) < 9: continue
        ci = int(parts[0]); xy = np.array(parts[1:], dtype=float).reshape(-1, 2) * (w, h)
        mask = np.zeros((h, w), np.uint8); cv2.fillPoly(mask, [xy.astype(np.int32)], 1)
        objs.append((ci, [int(xy[:, 0].min()), int(xy[:, 1].min()), int(xy[:, 0].max()), int(xy[:, 1].max())], mask))
    return objs

def box_iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    if ix2 <= ix1 or iy2 <= iy1: return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def mask_iou(a, b):
    return np.logical_and(a, b).sum() / (np.logical_or(a, b).sum() + 1e-9)

def ap_voc11(recalls, precisions):
    return sum(max((p for r, p in zip(recalls, precisions) if r >= t), default=0.) for t in np.arange(0, 1.1, 0.1)) / 11.

def average_precision(tp_confs, fp_confs, n_gt):
    if n_gt == 0 or not (tp_confs or fp_confs): return 0.0
    entries = sorted([(c, 1) for c in tp_confs] + [(c, 0) for c in fp_confs], key=lambda e: -e[0])
    tp = np.cumsum([e[1] for e in entries], dtype=float); fp = np.cumsum([1 - e[1] for e in entries], dtype=float)
    return ap_voc11((tp / n_gt).tolist(), (tp / (tp + fp + 1e-9)).tolist())

def evaluate(model, model_keys, img_dir, lbl_dir, class_names, conf_thr=0.20, iou_thr=0.5, imgsz=None):
    """Box/mask AP, instance mIoU and GT counts of `model` on a YOLO-seg folder.
    model_keys[i] is the PWVO key of the model's class i. Predictions of classes not in `class_names` are ignored."""
    local = {i: class_names.index(k) for i, k in enumerate(model_keys) if k in class_names}
    b_tp = {c: [] for c in class_names}; b_fp = {c: [] for c in class_names}
    m_tp = {c: [] for c in class_names}; m_fp = {c: [] for c in class_names}
    b_iou_v = {c: [] for c in class_names}; m_iou_v = {c: [] for c in class_names}
    n_gt = {c: 0 for c in class_names}; sem = []
    imgs = sorted(Path(img_dir).glob('*.jpg'))
    for img_f in tqdm(imgs, desc='eval', leave=False):
        gts = parse_yolo_seg_label(Path(lbl_dir) / (img_f.stem + '.txt'), img_f)
        for ci, _, _ in gts:
            if ci < len(class_names): n_gt[class_names[ci]] += 1
        kw = dict(conf=conf_thr, device=DEVICE, verbose=False)
        if imgsz: kw['imgsz'] = imgsz
        try: r = model.predict(str(img_f), **kw)[0]
        except Exception as e:
            print('predict failed on', img_f.name, '-', str(e)[:80]); continue
        H, W = r.orig_shape
        if r.boxes is None or len(r.boxes) == 0:
            if gts: sem.append(0.0)
            continue
        boxes = r.boxes.xyxy.cpu().numpy(); confs = r.boxes.conf.cpu().numpy(); cls = r.boxes.cls.cpu().numpy().astype(int)
        masks = r.masks.data.cpu().numpy() if r.masks is not None else None
        if masks is not None and masks.shape[1:] != (H, W):
            masks = np.stack([cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST) for m in masks])
        gt_union = np.zeros((H, W), bool)
        for _, _, gm in gts: gt_union |= gm.astype(bool)
        pred_union = np.zeros((H, W), bool)
        b_used, m_used = set(), set()
        for k in np.argsort(-confs):                     # same order for boxes and masks
            li = local.get(int(cls[k]), -1)
            if li < 0: continue
            name = class_names[li]; c = float(confs[k])
            hit = False
            for j, (gci, gb, gm) in enumerate(gts):
                if j in b_used or gci >= len(class_names) or class_names[gci] != name: continue
                v = box_iou(boxes[k], gb)
                if v >= iou_thr:
                    b_used.add(j); b_tp[name].append(c); b_iou_v[name].append(v); hit = True; break
            if not hit: b_fp[name].append(c)
            if masks is not None:
                pm = masks[k] > 0.5; pred_union |= pm; hit = False
                for j, (gci, gb, gm) in enumerate(gts):
                    if j in m_used or gci >= len(class_names) or class_names[gci] != name: continue
                    v = mask_iou(pm, gm.astype(bool))
                    if v >= iou_thr:
                        m_used.add(j); m_tp[name].append(c); m_iou_v[name].append(v); hit = True; break
                if not hit: m_fp[name].append(c)
        if gts or pred_union.any(): sem.append(mask_iou(pred_union, gt_union))

    box_ap  = {c: average_precision(b_tp[c], b_fp[c], n_gt[c]) for c in class_names}
    mask_ap = {c: average_precision(m_tp[c], m_fp[c], n_gt[c]) for c in class_names}
    est = [c for c in class_names if n_gt[c] > 0]
    mean = lambda d, cs: float(np.mean([d[c] for c in cs])) if cs else float('nan')
    return dict(n_images=len(imgs), n_gt=n_gt, estimable=est, box_ap=box_ap, mask_ap=mask_ap,
                box_map=mean(box_ap, est), mask_map=mean(mask_ap, est),
                box_map_raw=mean(box_ap, class_names), mask_map_raw=mean(mask_ap, class_names),
                box_miou={c: (float(np.mean(b_iou_v[c])) if b_iou_v[c] else None) for c in class_names},
                mask_miou={c: (float(np.mean(m_iou_v[c])) if m_iou_v[c] else None) for c in class_names},
                n_matched_mask={c: len(m_iou_v[c]) for c in class_names},
                semantic_miou=float(np.mean(sem)) if sem else None)

def split_means(res, covered, bu):
    """Mean box/mask AP over estimable classes of each split, plus harmonic mean of covered and BU."""
    out = {}
    for tag, ap in (('box', res['box_ap']), ('mask', res['mask_ap'])):
        for name, cs in (('covered', covered), ('BU', bu)):
            cs = [c for c in cs if c in res['estimable']]
            out[f'{tag}_{name}'] = float(np.mean([ap[c] for c in cs])) if cs else None
    return out
print('evaluator ready')

### 6.5 Table 9 — YOLOE-26 L and S on TACO

Both models use the full 47-category PWVO vocabulary (`set_vocab(model, ALL_KEYS)`), so class *i* is `ALL_KEYS[i]`; only the 14 TACO classes are scored. Results are written with the number of images and instances so the table can be checked against the subset actually used.

In [ ]:
TACO_CONF = 0.20
tab9, taco_runs = [], {}
for name, m in (('PlastiVision-Edge-L', model_l), ('PlastiVision-Edge-S', model_s)):
    keys = set_vocab(m, ALL_KEYS)
    res = evaluate(m, keys, TACO_IMGS, TACO_LBLS, TACO_BENCH_CATS, conf_thr=TACO_CONF)
    taco_runs[name] = res
    sp = split_means(res, TACO_COVERED, TACO_BU)
    tab9.append({'model': name, 'images': res['n_images'], 'instances': sum(res['n_gt'].values()),
                 'estimable_classes': len(res['estimable']),
                 'box_mAP_%': round(res['box_map'] * 100, 1), 'mask_mAP_%': round(res['mask_map'] * 100, 1),
                 'box_mAP_raw14_%': round(res['box_map_raw'] * 100, 1), 'mask_mAP_raw14_%': round(res['mask_map_raw'] * 100, 1),
                 'box_covered_%': round(sp['box_covered'] * 100, 1) if sp['box_covered'] is not None else None,
                 'box_BU_%': round(sp['box_BU'] * 100, 1) if sp['box_BU'] is not None else None})
df_tab9 = pd.DataFrame(tab9)
print(df_tab9.to_string(index=False))
df_tab9.to_csv(RES_DIR / 'table9_taco.csv', index=False)

res_L = taco_runs['PlastiVision-Edge-L']
per_class = pd.DataFrame({'class': TACO_BENCH_CATS,
    'split': ['covered' if PWVO[c]['covered'] else 'BU' for c in TACO_BENCH_CATS],
    'GT': [res_L['n_gt'][c] for c in TACO_BENCH_CATS],
    **{f'{n}_boxAP_%': [round(taco_runs[n]['box_ap'][c] * 100, 1) for c in TACO_BENCH_CATS] for n in taco_runs},
    **{f'{n}_maskAP_%': [round(taco_runs[n]['mask_ap'][c] * 100, 1) for c in TACO_BENCH_CATS] for n in taco_runs},
}).sort_values('GT', ascending=False)
print('\n', per_class.to_string(index=False))
per_class.to_csv(RES_DIR / 'table9_taco_per_class.csv', index=False)
print('\nclasses with 0 ground-truth instances are excluded from the mAP mean:', [c for c in TACO_BENCH_CATS if res_L['n_gt'][c] == 0])

### 6.6 Instance mIoU (companion to Table 9)

Mean IoU over *matched* prediction–ground-truth pairs (IoU ≥ 0.5), per class, from the same evaluation run. It is undefined for a class with no matched pair, so the number of matched pairs is reported next to every value; a macro-average over classes with few matches is a small, noisy sample. The image-level semantic mIoU (union of all predictions against union of all ground truth, averaged over images) is a different, usually higher quantity and is reported separately.

In [ ]:
rows = []
for name, res in taco_runs.items():
    est = [c for c in TACO_BENCH_CATS if res['mask_miou'][c] is not None]
    rows.append({'model': name,
                 'classes_with_matches': len(est),
                 'macro_mask_instance_mIoU': round(float(np.mean([res['mask_miou'][c] for c in est])), 4) if est else None,
                 'macro_box_instance_mIoU': round(float(np.mean([v for v in res['box_miou'].values() if v is not None])), 4) if any(v is not None for v in res['box_miou'].values()) else None,
                 'semantic_mIoU_all_images': round(res['semantic_miou'], 4) if res['semantic_miou'] is not None else None})
    pd.DataFrame({'class': TACO_BENCH_CATS, 'GT': [res['n_gt'][c] for c in TACO_BENCH_CATS],
                  'n_matched_mask_pairs': [res['n_matched_mask'][c] for c in TACO_BENCH_CATS],
                  'mask_instance_mIoU': [res['mask_miou'][c] for c in TACO_BENCH_CATS],
                  'box_instance_mIoU': [res['box_miou'][c] for c in TACO_BENCH_CATS]}
                 ).to_csv(RES_DIR / f"miou_{name.split('-')[-1].lower()}.csv", index=False)
df_miou = pd.DataFrame(rows); print(df_miou.to_string(index=False)); df_miou.to_csv(RES_DIR / 'miou_summary.csv', index=False)

## Part 7 — TrashNet plastic class

Whole-image *detection rate*: the fraction of the 482 TrashNet "plastic" photographs in which at least one of 10 target PWVO categories is detected at conf ≥ 0.10 with the 47-category vocabulary. There are no ground-truth boxes, so this is a lenient recall-style measure, not an AP, and its threshold (0.10) differs from the TACO evaluation (0.20).

Requires Kaggle credentials (`~/.kaggle/kaggle.json` or `KAGGLE_USERNAME`/`KAGGLE_KEY`), or download the dataset manually from <https://www.kaggle.com/datasets/feyzazkefe/trashnet> and unzip it to `data/trashnet/`.

In [ ]:
TRASH_DIR = DATA_DIR / 'trashnet'
if not (TRASH_DIR / 'dataset-resized' / 'plastic').exists():
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
    subprocess.check_call(['kaggle', 'datasets', 'download', '-d', 'feyzazkefe/trashnet', '-p', str(TRASH_DIR), '--unzip', '-q'])
trash_imgs = sorted((TRASH_DIR / 'dataset-resized' / 'plastic').glob('*.jpg'))
print(len(trash_imgs), 'TrashNet plastic images')

TRASH_TARGETS = {'pet_bottle', 'hdpe_bottle', 'plastic_bag', 'carrier_bag', 'eps_foam_cup',
                 'chip_packet', 'pp_food_container', 'pet_bottle_cap', 'cutlery', 'ldpe_film_sheet'}
TRASH_CONF = 0.10
trash_rows = []
for name, m in (('PlastiVision-Edge-S', model_s), ('PlastiVision-Edge-L', model_l)):
    keys = set_vocab(m, ALL_KEYS)
    hit = no_det = 0; per_cat = Counter(); top1 = Counter()
    for p in tqdm(trash_imgs, desc=name, leave=False):
        r = m.predict(str(p), conf=TRASH_CONF, device=DEVICE, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            no_det += 1; continue
        found = {keys[int(c)] for c in r.boxes.cls.tolist()}
        top1[keys[int(r.boxes.cls[r.boxes.conf.argmax()])]] += 1
        if found & TRASH_TARGETS:
            hit += 1; per_cat.update(found & TRASH_TARGETS)
    n = len(trash_imgs)
    trash_rows.append({'model': name, 'images': n, 'target_detected': hit, 'target_detected_%': round(hit / n * 100, 1),
                       'no_detection_at_all_%': round(no_det / n * 100, 1), 'conf_thr': TRASH_CONF})
    print(f'{name}: {hit}/{n} = {hit / n * 100:.1f}% | per category: {dict(per_cat.most_common())}')
    print('   top-1 predictions:', dict(top1.most_common(8)))
df_trash = pd.DataFrame(trash_rows); print(df_trash.to_string(index=False)); df_trash.to_csv(RES_DIR / 'trashnet_eval.csv', index=False)

## Part 8 — Prompt-design ablation on TACO (Table 5)

Three text-prompt strategies for the same 47 categories, YOLOE-26L, same evaluator and subset as Table 9: **generic** (the category name), **material-aware** (super-class + name + "waste"), and **full PWVO template** (the descriptive prompt). All 47 prompts are active in every arm so that class indices stay aligned with `ALL_KEYS`; only the text changes.

> The earlier version of this cell set a 14-prompt vocabulary but translated class indices with the 47-entry order, so predictions were credited to the wrong classes and the "full PWVO" arm scored about half of what the same prompts score in Table 9. It also had a fourth "prompt-free (LRPC)" arm that used exactly the same strings as the generic arm. Both problems are fixed here; a genuine prompt-free arm needs a mapping from the `-pf` model's built-in vocabulary to PWVO and is not included.

In [ ]:
def prompts_for(strategy):
    if strategy == 'generic':
        return [k.replace('_', ' ') for k in ALL_KEYS]
    if strategy == 'material-aware':
        return [f"{PWVO[k]['super'].split('/')[0].lower()} {k.replace('_', ' ')} waste" for k in ALL_KEYS]
    if strategy == 'full PWVO template':
        return [PWVO[k]['prompt'] for k in ALL_KEYS]
    raise ValueError(strategy)

abl = []
for strategy in ('generic', 'material-aware', 'full PWVO template'):
    pl = prompts_for(strategy)
    model_l.set_classes(pl, model_l.get_text_pe(pl)); force_model_to_device(model_l)
    res = evaluate(model_l, ALL_KEYS, TACO_IMGS, TACO_LBLS, TACO_BENCH_CATS, conf_thr=TACO_CONF)
    sp = split_means(res, TACO_COVERED, TACO_BU)
    abl.append({'strategy': strategy, 'images': res['n_images'],
                'box_mAP_%': round(res['box_map'] * 100, 1), 'mask_mAP_%': round(res['mask_map'] * 100, 1),
                'box_covered_%': round(sp['box_covered'] * 100, 1), 'box_BU_%': round(sp['box_BU'] * 100, 1) if sp['box_BU'] is not None else None})
df_abl = pd.DataFrame(abl); print(df_abl.to_string(index=False)); df_abl.to_csv(RES_DIR / 'table5_ablation.csv', index=False)

### 8b — Single-image prompt-confidence check (n = 1)

One photograph of a branded PET bottle, scored with a generic prompt and with the PWVO prompt. This is an anecdote, not a statistic: it needs a photograph in `data/samples/` named `bottle_*` and uses the best `pet_bottle` confidence at conf ≥ 0.001. The values are computed here, not typed in.

In [ ]:
bottle_files = [p for p in list_images(SMP_DIR) if p.name.lower().startswith('bottle')]
if not bottle_files:
    print('No data/samples/bottle_* photograph; skipping.')
else:
    GENERIC = 'clear transparent PET plastic water bottle discarded on ground'
    rows = []
    for label, prompt in (('generic', GENERIC), ('PWVO', PWVO['pet_bottle']['prompt'])):
        model_l.set_classes([prompt], model_l.get_text_pe([prompt])); force_model_to_device(model_l)
        for p in bottle_files:
            rows.append({'image': p.name, 'prompt': label, 'best_conf': round(best_conf(model_l, p, conf=0.001), 3)})
    df_pc = pd.DataFrame(rows); print(df_pc.to_string(index=False)); df_pc.to_csv(RES_DIR / 'prompt_confidence_check.csv', index=False)

## Part 9 — YOLO-World baseline (Table 9)

YOLO-World-S/L (`yolov8{s,l}-worldv2`) with the same 47 prompts, subset, evaluator and corrected mean as Table 9. YOLO-World has no segmentation head: mask AP and mIoU are *not applicable* (reported as N/A, not 0). Its text encoder is set once with `set_classes`; calling `set_classes` again after the model has been moved to the GPU fails with a device mismatch, so the vocabulary is set exactly once per model here.

In [ ]:
yw_rows, yw_models = [], {}
for tag, weights in (('YOLO-World-S', 'yolov8s-worldv2.pt'), ('YOLO-World-L', 'yolov8l-worldv2.pt')):
    m = YOLO(weights)
    m.set_classes(ALL_PROMPTS)            # CLIP text encoder runs on CPU here; move to GPU afterwards
    m.model.to(DEVICE)
    res = evaluate(m, ALL_KEYS, TACO_IMGS, TACO_LBLS, TACO_BENCH_CATS, conf_thr=TACO_CONF)
    sp = split_means(res, TACO_COVERED, TACO_BU)
    yw_rows.append({'model': tag, 'images': res['n_images'], 'box_mAP_%': round(res['box_map'] * 100, 1), 'mask_mAP_%': 'N/A',
                    'box_covered_%': round(sp['box_covered'] * 100, 1), 'box_BU_%': round(sp['box_BU'] * 100, 1) if sp['box_BU'] is not None else None})
    yw_models[tag] = m
df_yw = pd.DataFrame(yw_rows); print(df_yw.to_string(index=False)); df_yw.to_csv(RES_DIR / 'table9_yolo_world.csv', index=False)

## Part 10 — Plastic-coverage estimation

Coverage = fraction of image pixels covered by the union of predicted masks (YOLOE-26L, 47 categories, conf ≥ 0.20). With the polystyrene prompts the model occasionally returns a near-full-frame, low-confidence mask ("EPS attractor"); the estimator therefore **drops any detection whose box covers more than 80 % of the frame unless its confidence is at least 0.60**, and reports how often that filter fires. Both numbers (filtered coverage, filter firing rate) are results; the 80 % / 0.60 thresholds were chosen after inspecting the failure cases and are not tuned on a held-out set.

In [ ]:
MAX_BOX_AREA_PCT, MIN_CONF_LARGE_BOX, COV_CONF = 80.0, 0.60, 0.20

def estimate_coverage(img_path, model, keys, conf_thr=COV_CONF, max_area_pct=MAX_BOX_AREA_PCT, min_conf_large=MIN_CONF_LARGE_BOX):
    """Returns (total_pct, {super: pct}, n_dropped). keys[i] is the PWVO key of model class i."""
    H, W = cv2.imread(str(img_path)).shape[:2]
    r = model.predict(str(img_path), conf=conf_thr, device=DEVICE, verbose=False)[0]
    per_super = {s: np.zeros((H, W), np.uint8) for s in SUPER_COLORS}; union = np.zeros((H, W), np.uint8); dropped = 0
    if r.masks is None or r.boxes is None or len(r.boxes) == 0:
        return 0.0, {s: 0.0 for s in SUPER_COLORS}, dropped
    masks = r.masks.data.cpu().numpy(); boxes = r.boxes.xyxy.cpu().numpy()
    for i in range(min(len(r.boxes), len(masks))):
        conf = float(r.boxes.conf[i])
        if (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1]) / (H * W) * 100 > max_area_pct and conf < min_conf_large:
            dropped += 1; continue
        sup = PWVO[keys[int(r.boxes.cls[i])]]['super']
        m = masks[i] if masks[i].shape == (H, W) else cv2.resize(masks[i], (W, H), interpolation=cv2.INTER_LINEAR)
        b = (m > 0.5).astype(np.uint8); union = np.maximum(union, b); per_super[sup] = np.maximum(per_super[sup], b)
    return union.sum() / (H * W) * 100, {s: v.sum() / (H * W) * 100 for s, v in per_super.items()}, dropped

keys_L = set_vocab(model_l, ALL_KEYS)
cov_rows = []
for p in tqdm(sorted(TACO_IMGS.glob('*.jpg')), desc='coverage'):
    try: tot, per, dr = estimate_coverage(p, model_l, keys_L)
    except Exception as e: print('skip', p.name, str(e)[:60]); continue
    cov_rows.append({'image': p.name, 'total_pct': tot, 'n_dropped': dr, **{f'super_{k}': v for k, v in per.items()}})
cov = pd.DataFrame(cov_rows); cov.to_csv(RES_DIR / 'coverage_taco_filtered.csv', index=False)
nz = cov[cov.total_pct > 0]
print(f'images: {len(cov)} | filter fired on {(cov.n_dropped > 0).sum()} ({(cov.n_dropped > 0).mean() * 100:.1f}%), {int(cov.n_dropped.sum())} boxes dropped')
print(f'mean coverage, all images: {cov.total_pct.mean():.2f} +- {cov.total_pct.std():.2f} %')
print(f'mean coverage, images with any coverage: {nz.total_pct.mean():.2f} +- {nz.total_pct.std():.2f} %  (n={len(nz)})')

## Part 11 — Edge export and latency

### 11.1 Export (YOLOE-26S with the 47 PWVO prompts baked in)

`export()` re-reads the checkpoint, so the prompts are baked in, the model is **saved and reloaded**, and only then exported. Formats: ONNX, TFLite FP32, TFLite INT8 (YOLOE-26N: ONNX and TFLite FP32). Sizes are reported in KiB and MiB (1 MiB = 1,048,576 bytes); the Ultralytics log prints MiB, so decimal-MB figures from other tools will look about 5 % larger. INT8 quantisation uses Ultralytics' default calibration data; set `data=` in `do_export` to calibrate on PWVO images instead. CoreML needs macOS and is skipped elsewhere. Needs Python ≥ 3.12 for the nested-quote f-string-free code below only if you edit it; the code itself runs on 3.10+.

In [ ]:
import platform
pwvo_names = list(PWVO)
assert len(pwvo_names) == 47

m_s = load_yoloe('yoloe-26s-seg.pt')
set_vocab(m_s, ALL_KEYS)
print('classes in head after set_classes:', m_s.model.model[-1].nc)
m_s.save(str(EXP_DIR / 'yoloe-26s-seg-pwvo47.pt'))          # save + reload: export() otherwise uses the original head
m_s = load_yoloe(str(EXP_DIR / 'yoloe-26s-seg-pwvo47.pt'))
assert m_s.model.model[-1].nc == 47, m_s.model.model[-1].nc
(EXP_DIR / 'pwvo_labels.json').write_text(json.dumps(pwvo_names, indent=2))   # label order for the Android app

def find_tflite(out, int8=False):
    base = Path(str(out))
    if base.is_file() and base.suffix == '.tflite': return base
    pats = (['**/*integer_quant*.tflite', '**/*int8*.tflite', '**/*.tflite'] if int8
            else ['**/*float32*.tflite', '**/*.tflite'])
    for pat in pats:
        hits = [h for h in glob.glob(str(base / pat), recursive=True) if Path(h).is_file()]
        if hits: return Path(hits[0])
    return None

exp_rows = []
def do_export(model, fmt, dest_name, int8=False, imgsz=640):
    tag = f"{fmt} {'INT8' if int8 else 'FP32'}"
    try:
        t0 = time.time(); out = model.export(format=fmt, imgsz=imgsz, int8=int8, device='cpu'); dt = time.time() - t0
        p = find_tflite(out, int8) if fmt == 'tflite' else Path(str(out))
        if p is None or not p.is_file(): raise FileNotFoundError(f'no output file in {out}')
        dst = EXP_DIR / dest_name; shutil.copy2(p, dst)
        row = {'model': dest_name, 'format': tag, 'size_KiB': round(dst.stat().st_size / 1024), 'size_MiB': round(dst.stat().st_size / 2**20, 1),
               'export_s': round(dt), 'status': 'OK'}
    except Exception as e:
        row = {'model': dest_name, 'format': tag, 'size_KiB': None, 'size_MiB': None, 'export_s': None, 'status': str(e)[:80]}
    print(row); exp_rows.append(row)

do_export(m_s, 'onnx',   'yoloe_26s.onnx')
do_export(m_s, 'tflite', 'yoloe_26s_fp32.tflite')
do_export(m_s, 'tflite', 'yoloe_26s_int8.tflite', int8=True)
m_n = load_yoloe('yoloe-26n-seg.pt'); set_vocab(m_n, ALL_KEYS)
do_export(m_n, 'onnx',   'yoloe_26n.onnx')
do_export(m_n, 'tflite', 'yoloe_26n_fp32.tflite')
if platform.system() == 'Darwin':
    do_export(m_s, 'coreml', 'yoloe_26s.mlpackage')
df_exp = pd.DataFrame(exp_rows); print(df_exp.to_string(index=False)); df_exp.to_csv(RES_DIR / 'export_summary.csv', index=False)

In [ ]:
# Model scale (measured): parameters of the unfused PyTorch models, and exported file sizes from above
scale = []
for tag, w in (('YOLOE-26N', 'yoloe-26n-seg.pt'), ('YOLOE-26S', 'yoloe-26s-seg.pt'), ('YOLOE-26M', 'yoloe-26m-seg.pt'), ('YOLOE-26L', 'yoloe-26l-seg.pt')):
    try: scale.append({'model': tag, 'params_M_unfused': round(n_params_m(load_yoloe(w)), 1)})
    except Exception as e: scale.append({'model': tag, 'params_M_unfused': None})
df_scale = pd.DataFrame(scale); print(df_scale.to_string(index=False)); df_scale.to_csv(RES_DIR / 'model_scale.csv', index=False)
print('Note: Ultralytics prints the fused parameter count in its model summary (fewer parameters, BatchNorm folded into Conv); state which one a table uses.')

### 11.2 GPU latency

End-to-end `predict()` latency (pre-processing, inference, mask decoding, post-processing) on a 640×640 frame, batch 1, 8 warm-up runs and 40 timed runs per trial, 5 trials; the table reports the mean and the standard deviation **across trials**, and lists every individual trial so that no trial is silently excluded (the first trials after a vocabulary change can be slower). The numbers are specific to the GPU printed in Part 1. YOLO-World is measured on the same GPU rather than quoted from the literature.

Phone latencies come from the Android app, not from this notebook: record them in `data/android_latency.csv` (columns `label,latency_ms`) and they are added to the figure.

In [ ]:
def profile_once(model, frame_path, n_warm=8, n_runs=40):
    kw = dict(source=str(frame_path), conf=0.20, device=DEVICE, imgsz=640, verbose=False)
    for _ in range(n_warm): model.predict(**kw)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); model.predict(**kw)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(ts))

lat_frame = ROOT / 'tmp_latency_frame.jpg'
src = (list_images(SMP_DIR) or list_images(ZS_DIR) or [None])[0]
assert src is not None, f'put a photograph in {SMP_DIR} or {ZS_DIR} to profile on'
cv2.imwrite(str(lat_frame), cv2.resize(cv2.imread(str(src)), (640, 640)))

N_TRIALS = 5
lat_models = [('YOLOE-26S + PWVO (text)', model_s, 'text'), ('YOLOE-26L + PWVO (text)', model_l, 'text'),
              ('YOLOE-26S prompt-free', model_pf, None)]
if 'yw_models' in globals():
    lat_models += [(f'{k} + PWVO (text)', v, None) for k, v in yw_models.items()]     # vocabulary already set in Part 9
lat_rows = []
for name, m, vocab in lat_models:
    if vocab == 'text': set_vocab(m, ALL_KEYS)
    trials = [profile_once(m, lat_frame) for _ in range(N_TRIALS)]
    lat_rows.append({'model': name, 'device': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU',
                     'latency_ms_mean': round(np.mean(trials), 1), 'latency_ms_sd_across_trials': round(np.std(trials, ddof=1), 1),
                     'fps': round(1000 / np.mean(trials), 1), 'trials_ms': ' '.join(f'{t:.1f}' for t in trials), 'masks': 'no' if 'World' in name else 'yes'})
    print(lat_rows[-1])
lat_frame.unlink(missing_ok=True)
df_lat = pd.DataFrame(lat_rows); df_lat.to_csv(RES_DIR / 'latency_gpu.csv', index=False)

In [ ]:
# Latency figure: measured GPU rows + any phone rows recorded in data/android_latency.csv
entries = [(r['model'].replace(' + PWVO (text)', '').replace(' ', '\n', 1), r['latency_ms_mean'], '#1976D2') for r in lat_rows]
android_csv = DATA_DIR / 'android_latency.csv'
if android_csv.exists():
    entries += [(r.label, r.latency_ms, '#388E3C') for r in pd.read_csv(android_csv).itertuples()]
labels, vals, cols = zip(*entries)
fig, ax = plt.subplots(figsize=(12, 4.5))
bars = ax.bar(range(len(vals)), vals, color=cols, edgecolor='white'); ax.set_yscale('log')
ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels, fontsize=8); ax.set_ylabel('end-to-end latency (ms, log)')
for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, v * 1.08, f'{v:g}', ha='center', fontsize=8, fontweight='bold')
ax.legend(handles=[mpatches.Patch(color='#1976D2', label='GPU (measured here)'), mpatches.Patch(color='#388E3C', label='Android (data/android_latency.csv)')])
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(RES_DIR / 'latency.png', dpi=150, facecolor='white'); plt.show()

## Appendix A — Generators for the qualitative figures that use the authors' photographs

These cells reproduce the paper's hand-picked qualitative figures (seen-vs-unseen panels, the South-Asian packaging examples, and the coverage example). They need the authors' own photographs, which are listed in `data/README.md`; they are **not** needed for any table. Paths were changed from `/content/...` to the folders configured in Part 1, and browser downloads were removed; the drawing code is otherwise as it was when the figures were produced.

In [ ]:
FIG_DIR = DATA_DIR / 'figure_images'      # the authors' photographs used in the qualitative figures
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# COMPLETE CELL — Run this entirely fresh
# Produces:
#   fig_seen_vs_unseen.png     (Fig 3 — two panel)
#   fig_novel_aavin.png        (Novel — Aavin brand vs material)
#   fig_novel_maggi.png        (Novel — Maggi brand vs material)
# ═══════════════════════════════════════════════════════════════════

import cv2, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path

ZS_DIR = FIG_DIR

img_a_path = ZS_DIR / 'pet_bottle.jpg'
img_b_path = ZS_DIR / 'eps_block.jpg'

aavin_path = str(FIG_DIR / 'aavin.jpg')
maggi_path = str(FIG_DIR / 'maggi.jpg')

# ── Colour map ────────────────────────────────────────────────────
SUPER_COLORS = {
    'PET/Polyester':        '#1D9E75',
    'HDPE/LDPE film':       '#378ADD',
    'Polypropylene':        '#D85A30',
    'Polystyrene/EPS':      '#BA7517',
    'Multilayer/composite': '#D4537E',
    'Other':                '#888780',
}

# ── Shared draw function ──────────────────────────────────────────
def draw_panel(ax, img_rgb, result, pwvo_keys, pwvo,
               mode='all', conf_thr=0.15):
    ax.imshow(img_rgb)
    ax.set_xticks([]); ax.set_yticks([])
    H_i, W_i = img_rgb.shape[:2]

    if result.boxes is None or len(result.boxes) == 0:
        ax.text(W_i//2, H_i//2, 'No detections above threshold',
                ha='center', va='center', fontsize=11,
                color='red', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4',
                          fc='white', alpha=0.85))
        return

    pboxes = result.boxes.xyxy.cpu().numpy()
    pconfs = result.boxes.conf.cpu().numpy()
    pcls   = result.boxes.cls.cpu().numpy().astype(int)
    pmasks = result.masks.data.cpu().numpy() \
             if result.masks is not None else None

    if pmasks is not None and pmasks.shape[1:] != (H_i, W_i):
        pmasks = np.stack([
            cv2.resize(pmasks[k], (W_i, H_i),
                       interpolation=cv2.INTER_NEAREST)
            for k in range(len(pmasks))
        ])

    for k, (pb, pc, pci) in enumerate(
        sorted(zip(pboxes, pconfs, pcls), key=lambda x: -x[1])
    ):
        if pci >= len(pwvo_keys): continue
        if pc < conf_thr: continue
        label    = pwvo_keys[pci]
        seen     = pwvo.get(label, {}).get('seen', True)
        super_c  = pwvo.get(label, {}).get('super', 'Other')
        color    = SUPER_COLORS.get(super_c, '#888780')
        split_tag = 'SEEN' if seen else 'UNSEEN'

        if mode == 'seen'   and not seen: continue
        if mode == 'unseen' and seen:     continue

        # Mask overlay
        if pmasks is not None and k < len(pmasks):
            overlay = np.zeros((H_i, W_i, 4))
            rgba = list(mcolors.to_rgba(color)); rgba[3] = 0.42
            overlay[pmasks[k] > 0.5] = rgba
            ax.imshow(overlay)

        # Bounding box
        x1, y1, x2, y2 = pb
        rect = plt.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)

        # Label — reduced fontsize to prevent truncation
        ax.text(x1, max(y1-5, 14),
                f"{label}  {pc:.2f}  [{split_tag}]",
                color='white', fontsize=7.5, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.25',
                          fc=color, alpha=0.90))


# ═════════════════════════════════════════════════════════════════
# FIGURE 1 — Two-panel: Seen (A) + Unseen (B)
# ═════════════════════════════════════════════════════════════════

full_prompts = [v['prompt'] for v in PWVO.values()]
text_pe = model_l.get_text_pe(full_prompts)
model_l.set_classes(full_prompts, text_pe)
pwvo_keys = list(PWVO.keys())

r_a = model_l.predict(str(img_a_path), conf=0.15, verbose=False)[0]
r_b = model_l.predict(str(img_b_path), conf=0.15, verbose=False)[0]

img_a = cv2.cvtColor(cv2.imread(str(img_a_path)), cv2.COLOR_BGR2RGB)
img_b = cv2.cvtColor(cv2.imread(str(img_b_path)), cv2.COLOR_BGR2RGB)

print("\n── Panel A detections (pet_bottle, conf≥0.15) ──")
if r_a.boxes is not None:
    for conf, cls in zip(r_a.boxes.conf.tolist(),
                         r_a.boxes.cls.tolist()):
        lbl   = pwvo_keys[int(cls)]
        split = 'SEEN' if PWVO[lbl]['seen'] else 'UNSEEN'
        print(f"  {lbl:<25} conf={conf:.3f}  [{split}]")

print("\n── Panel B detections (eps_block, conf≥0.15) ──")
if r_b.boxes is not None:
    for conf, cls in zip(r_b.boxes.conf.tolist(),
                         r_b.boxes.cls.tolist()):
        lbl   = pwvo_keys[int(cls)]
        split = 'SEEN' if PWVO[lbl]['seen'] else 'UNSEEN'
        print(f"  {lbl:<25} conf={conf:.3f}  [{split}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.subplots_adjust(wspace=0.05, left=0.01, right=0.99,
                    top=0.85, bottom=0.10)

draw_panel(axes[0], img_a, r_a, pwvo_keys, PWVO,
           mode='seen', conf_thr=0.15)
axes[0].set_title(
    '(A)  Seen-category detection\n'
    'pet_bottle  [SEEN · PET/Polyester]  ·  YOLOE-26L + full PWVO',
    fontsize=10.5, color='#1D9E75', fontweight='bold', pad=7
)
for sp in axes[0].spines.values():
    sp.set_edgecolor('#1D9E75'); sp.set_linewidth(3)

draw_panel(axes[1], img_b, r_b, pwvo_keys, PWVO,
           mode='unseen', conf_thr=0.15)
axes[1].set_title(
    '(B)  Unseen zero-shot generalisation\n'
    'eps_block  [UNSEEN · Polystyrene/EPS]  ·  YOLOE-26L + full PWVO',
    fontsize=10.5, color='#BA7517', fontweight='bold', pad=7
)
for sp in axes[1].spines.values():
    sp.set_edgecolor('#BA7517'); sp.set_linewidth(3)

legend_items = [
    mpatches.Patch(color='#1D9E75', label='PET/Polyester'),
    mpatches.Patch(color='#378ADD', label='HDPE/LDPE film'),
    mpatches.Patch(color='#D85A30', label='Polypropylene'),
    mpatches.Patch(color='#BA7517', label='Polystyrene/EPS'),
    mpatches.Patch(color='#D4537E', label='Multilayer/composite'),
    mpatches.Patch(color='#888780', label='Other'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=6,
           fontsize=9.5, frameon=True, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01))

fig.suptitle(
    'PlastiVision-Edge — Seen vs. Unseen Zero-Shot Detection\n'
    'YOLOE-26L · Full 47-class PWVO vocabulary · '
    'No retraining · conf ≥ 0.15',
    fontsize=11, fontweight='bold', y=0.97
)

out_ab = RES_DIR / 'fig_seen_vs_unseen.png'
plt.savefig(out_ab, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n✅ Saved → {out_ab}")


# ═════════════════════════════════════════════════════════════════
# HELPER — Novel class two-panel figure generator
# Used for both Aavin and Maggi
# ═════════════════════════════════════════════════════════════════

def run_novel_figure(img_path, prompts_dict, item_name,
                     out_filename, conf_thr=0.05):
    """
    prompts_dict = {
        'brand_name': '...',
        'material_descriptive': '...'
    }
    Produces a two-panel figure and prints detection summary.
    """
    img_bgr  = cv2.imread(img_path)
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H_n, W_n = img_rgb.shape[:2]

    PANEL_META = {
        'brand_name': {
            'color': '#C0392B',
            'label': f'(A)  Brand-name prompt\n"{prompts_dict["brand_name"]}"'
        },
        'material_descriptive': {
            'color': '#1D6FA4',
            'label': f'(B)  Material-descriptive prompt\n'
                     f'"{prompts_dict["material_descriptive"]}"'
        },
    }

    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 7))
    fig2.subplots_adjust(wspace=0.05, left=0.01, right=0.99,
                         top=0.82, bottom=0.06)

    results_summary = {}

    for ax, (pkey, prompt) in zip(axes2, prompts_dict.items()):
        text_pe_n = model_l.get_text_pe([prompt])
        model_l.set_classes([prompt], text_pe_n)
        r_n = model_l.predict(img_path, conf=conf_thr,
                              verbose=False)[0]

        color = PANEL_META[pkey]['color']
        title = PANEL_META[pkey]['label']

        ax.imshow(img_rgb)
        ax.set_xticks([]); ax.set_yticks([])

        detections = []
        if r_n.boxes is not None and len(r_n.boxes) > 0:
            pboxes = r_n.boxes.xyxy.cpu().numpy()
            pconfs = r_n.boxes.conf.cpu().numpy()
            pmasks = r_n.masks.data.cpu().numpy() \
                     if r_n.masks is not None else None

            if pmasks is not None and pmasks.shape[1:] != (H_n, W_n):
                pmasks = np.stack([
                    cv2.resize(pmasks[k], (W_n, H_n),
                               interpolation=cv2.INTER_NEAREST)
                    for k in range(len(pmasks))
                ])

            for k, (pb, pc) in enumerate(
                sorted(zip(pboxes, pconfs), key=lambda x: -x[1])
            ):
                if pc < conf_thr: continue
                detections.append(pc)

                # Mask
                if pmasks is not None and k < len(pmasks):
                    ov = np.zeros((H_n, W_n, 4))
                    rgba = list(mcolors.to_rgba(color))
                    rgba[3] = 0.42
                    ov[pmasks[k] > 0.5] = rgba
                    ax.imshow(ov)

                # Box
                x1, y1, x2, y2 = pb
                rect = plt.Rectangle(
                    (x1, y1), x2-x1, y2-y1,
                    linewidth=2.5, edgecolor=color,
                    facecolor='none'
                )
                ax.add_patch(rect)

                # Label
                ax.text(x1, max(y1-5, 14),
                        f'conf={pc:.3f}',
                        color='white', fontsize=10,
                        fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.25',
                                  fc=color, alpha=0.90))

        if not detections:
            ax.text(W_n//2, H_n//2, 'Not detected',
                    ha='center', va='center', fontsize=13,
                    color='red', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.4',
                              fc='white', alpha=0.85))

        results_summary[pkey] = detections
        ax.set_title(title, fontsize=10, color=color,
                     fontweight='bold', pad=7)
        for sp in ax.spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(3)

    # Print summary
    print(f"\n── {item_name} detection summary ──")
    for pkey, dets in results_summary.items():
        if dets:
            print(f"  [{pkey}]  ✅ {len(dets)} detection(s)  "
                  f"max conf={max(dets):.3f}  "
                  f"all={[f'{c:.3f}' for c in dets]}")
        else:
            print(f"  [{pkey}]  ❌ not detected")

    # Determine suptitle based on results
    brand_dets  = results_summary['brand_name']
    mat_dets    = results_summary['material_descriptive']
    brand_max   = max(brand_dets)  if brand_dets  else 0.0
    mat_max     = max(mat_dets)    if mat_dets    else 0.0

    if not brand_dets and mat_dets:
        narrative = (f"Brand-name prompt fails (conf=0.000) · "
                     f"Material-descriptive prompt succeeds "
                     f"(conf={mat_max:.3f})")
    elif brand_dets and mat_dets and mat_max > brand_max:
        narrative = (f"Material-descriptive prompt (conf={mat_max:.3f}) "
                     f"outperforms brand-name prompt (conf={brand_max:.3f})")
    elif brand_dets and mat_dets and brand_max >= mat_max:
        narrative = (f"Both prompts detect — brand (conf={brand_max:.3f}), "
                     f"material-descriptive (conf={mat_max:.3f})")
    else:
        narrative = "Neither prompt achieves reliable detection"

    fig2.suptitle(
        f'PlastiVision-Edge — Novel Open-Vocabulary Detection: '
        f'{item_name}\n{narrative}',
        fontsize=11, fontweight='bold', y=0.97
    )

    out_path = RES_DIR / out_filename
    plt.savefig(out_path, dpi=150, bbox_inches='tight',
                facecolor='white')
    plt.show()
    print(f"\n✅ Saved → {out_path}")
    return results_summary


# ═════════════════════════════════════════════════════════════════
# FIGURE 2 — Aavin milk packet
# ═════════════════════════════════════════════════════════════════

aavin_results = run_novel_figure(
    img_path   = aavin_path,
    item_name  = 'Aavin Milk Packet',
    out_filename = 'fig_novel_aavin.png',
    prompts_dict = {
        'brand_name':
            'Aavin milk packet',
        'material_descriptive':
            'sealed flat flexible LDPE plastic milk pouch '
            'with coloured printed branding',
    },
    conf_thr = 0.05
)


# ═════════════════════════════════════════════════════════════════
# FIGURE 3 — Maggi noodles packet
# ═════════════════════════════════════════════════════════════════

maggi_results = run_novel_figure(
    img_path   = maggi_path,
    item_name  = 'Maggi Noodles Packet',
    out_filename = 'fig_novel_maggi.png',
    prompts_dict = {
        'brand_name':
            'Maggi noodles packet',
        'material_descriptive':
            'yellow flexible multilayer laminate food packet '
            'with printed label and sealed edges',
    },
    conf_thr = 0.05
)


# ═════════════════════════════════════════════════════════════════
# COMBINED SUMMARY — decide which item to use in the paper
# ═════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("NOVEL CLASS FIGURE SELECTION SUMMARY")
print("="*60)

for name, res in [('Aavin', aavin_results),
                  ('Maggi', maggi_results)]:
    b = max(res['brand_name'])         if res['brand_name']         else 0.0
    m = max(res['material_descriptive']) if res['material_descriptive'] else 0.0
    gap = m - b
    print(f"\n{name}:")
    print(f"  Brand-name prompt       : {'✅' if b > 0 else '❌'} "
          f"conf={b:.3f}")
    print(f"  Material-descriptive    : {'✅' if m > 0 else '❌'} "
          f"conf={m:.3f}")
    print(f"  Gap (material - brand)  : {gap:+.3f}")
    if b == 0 and m > 0:
        print(f"  → brand prompt fails, material prompt succeeds")
    elif gap > 0.1:
        print(f"  → material prompt clearly stronger")
    elif gap > 0:
        print(f"  → WEAK: material only slightly better")
    else:
        print(f"  → brand prompt equals or beats material")

In [ ]:
# ── Regenerate Fig 6 — South Asian + Novel categories ─────────────
# Panel A: Maggi noodles packet (replaces Aavin)
#           material-descriptive prompt → conf=0.909
# Panel B: Gold multilayer retort pouch
# Panel C: Aluminium blister pack

import cv2, numpy as np, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path


# ── Load images ───────────────────────────────────────────────────
# Maggi — use the same image from the novel figure run
maggi_path   = str(FIG_DIR / 'maggi.jpeg')
retort_path  = str(FIG_DIR / 'gold_retort.jpg')
blister_path = str(FIG_DIR / 'blister_pack.jpg')

# ── Prompts ───────────────────────────────────────────────────────
MAGGI_PROMPT   = 'yellow flexible multilayer laminate food packet with printed label and sealed edges'
RETORT_PROMPT  = PWVO['retort_pouch']['prompt']   # multilayer film pouch for food storage
BLISTER_PROMPT = PWVO['blister_pack']['prompt']

SUPER_COLORS = {
    'PET/Polyester':        '#1D9E75',
    'HDPE/LDPE film':       '#378ADD',
    'Polypropylene':        '#D85A30',
    'Polystyrene/EPS':      '#BA7517',
    'Multilayer/composite': '#D4537E',
    'Other':                '#888780',
}

def run_single_prompt(model, img_path, prompt, conf_thr=0.10):
    """Run inference with a single prompt, return result."""
    text_pe = model.get_text_pe([prompt])
    model.set_classes([prompt], text_pe)
    r = model.predict(img_path, conf=conf_thr, verbose=False)[0]
    return r

def draw_detections_single(ax, img_rgb, result, color,
                            label_text=None, conf_thr=0.10):
    """Draw boxes and masks for single-prompt result."""
    ax.imshow(img_rgb)
    ax.set_xticks([]); ax.set_yticks([])
    H, W = img_rgb.shape[:2]

    if result.boxes is None or len(result.boxes) == 0:
        ax.text(W//2, H//2, 'Not detected',
                ha='center', va='center', fontsize=11,
                color='red', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4',
                          fc='white', alpha=0.85))
        return

    pboxes = result.boxes.xyxy.cpu().numpy()
    pconfs = result.boxes.conf.cpu().numpy()
    pmasks = result.masks.data.cpu().numpy() \
             if result.masks is not None else None

    if pmasks is not None and pmasks.shape[1:] != (H, W):
        pmasks = np.stack([
            cv2.resize(pmasks[k], (W, H),
                       interpolation=cv2.INTER_NEAREST)
            for k in range(len(pmasks))
        ])

    for k, (pb, pc) in enumerate(
        sorted(zip(pboxes, pconfs), key=lambda x: -x[1])
    ):
        if pc < conf_thr: continue

        if pmasks is not None and k < len(pmasks):
            overlay = np.zeros((H, W, 4))
            rgba = list(mcolors.to_rgba(color)); rgba[3] = 0.42
            overlay[pmasks[k] > 0.5] = rgba
            ax.imshow(overlay)

        x1, y1, x2, y2 = pb
        rect = plt.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, max(y1-5, 14),
                f'conf={pc:.3f}',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.25',
                          fc=color, alpha=0.90))

# ── Load images ───────────────────────────────────────────────────
img_maggi   = cv2.cvtColor(cv2.imread(maggi_path),   cv2.COLOR_BGR2RGB)
img_retort  = cv2.cvtColor(cv2.imread(retort_path),  cv2.COLOR_BGR2RGB)
img_blister = cv2.cvtColor(cv2.imread(blister_path), cv2.COLOR_BGR2RGB)

# ── Run inference ─────────────────────────────────────────────────
# Panel A — Maggi: material-descriptive prompt on model_l
print("Panel A — Maggi (material-descriptive prompt, YOLOE-26L)...")
r_maggi = run_single_prompt(model_l, maggi_path,
                             MAGGI_PROMPT, conf_thr=0.05)
maggi_confs = r_maggi.boxes.conf.cpu().tolist() \
              if r_maggi.boxes is not None else []
print(f"  Detections: {len(maggi_confs)}  "
      f"max conf={max(maggi_confs):.3f}" if maggi_confs else "  Not detected")

# Panel B — Gold retort pouch: YOLOE-26L
print("Panel B — Gold retort pouch (YOLOE-26L)...")
r_retort_l = run_single_prompt(model_l, retort_path,
                                RETORT_PROMPT, conf_thr=0.10)
retort_confs = r_retort_l.boxes.conf.cpu().tolist() \
               if r_retort_l.boxes is not None else []
print(f"  YOLOE-26L: max conf={max(retort_confs):.3f}" \
      if retort_confs else "  YOLOE-26L: Not detected")

# Panel C — Blister pack: YOLOE-26S
print("Panel C — Blister pack (YOLOE-26S)...")
r_blister_s = run_single_prompt(model_s, blister_path,
                                 BLISTER_PROMPT, conf_thr=0.10)
blister_confs = r_blister_s.boxes.conf.cpu().tolist() \
                if r_blister_s.boxes is not None else []
n_blister = len(blister_confs)
print(f"  YOLOE-26S: {n_blister} detections  "
      f"max conf={max(blister_confs):.3f}" if blister_confs \
      else "  YOLOE-26S: Not detected")

# ── Build figure ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.subplots_adjust(wspace=0.04, left=0.01, right=0.99,
                    top=0.84, bottom=0.04)

MULTI_COLOR = SUPER_COLORS['Multilayer/composite']  # #D4537E

# Panel A — Maggi
draw_detections_single(axes[0], img_maggi, r_maggi,
                       MULTI_COLOR, conf_thr=0.05)
axes[0].set_title(
    'Maggi Noodles Packet (Nestlé India)\n'
    'Material-descriptive prompt · YOLOE-26L\n'
    f'conf = {max(maggi_confs):.3f}  |  YOLO-World-S: 0 detections',
    fontsize=9, color=MULTI_COLOR, fontweight='bold', pad=6
)
for sp in axes[0].spines.values():
    sp.set_edgecolor(MULTI_COLOR); sp.set_linewidth(3)

# Panel B — Gold retort pouch
draw_detections_single(axes[1], img_retort, r_retort_l,
                       MULTI_COLOR, conf_thr=0.10)
axes[1].set_title(
    'Gold Multilayer Retort Pouch\n'
    f'YOLOE-26L conf = {max(retort_confs):.3f}  |  '
    f'YOLO-World-S: 0 detections',
    fontsize=9, color=MULTI_COLOR, fontweight='bold', pad=6
)
for sp in axes[1].spines.values():
    sp.set_edgecolor(MULTI_COLOR); sp.set_linewidth(3)

# Panel C — Blister pack
draw_detections_single(axes[2], img_blister, r_blister_s,
                       SUPER_COLORS['Multilayer/composite'],
                       conf_thr=0.10)
axes[2].set_title(
    f'Aluminium Blister Pack\n'
    f'YOLOE-26S: {n_blister} detections  '
    f'max conf = {max(blister_confs):.3f}  |  '
    f'YOLOE-26L + YOLO-World: 0 detections',
    fontsize=9, color=MULTI_COLOR, fontweight='bold', pad=6
)
for sp in axes[2].spines.values():
    sp.set_edgecolor(MULTI_COLOR); sp.set_linewidth(3)

fig.suptitle(
    'YOLOE detects South Asian packaging and novel categories '
    'that YOLO-World-S misses entirely\n'
    'No retraining · Vocabulary set at inference time via CLIP embeddings',
    fontsize=11, fontweight='bold', y=0.97
)

out = RES_DIR / 'fig_south_asian_novel.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n✅ Saved → {out}")

In [ ]:
# ── Regenerate Fig 4 — Coverage Estimation (fixed bar chart) ──────
import cv2, numpy as np, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

img_path = str(FIG_DIR / 'pet_bottle.jpg')

full_prompts = [v['prompt'] for v in PWVO.values()]
text_pe = model_l.get_text_pe(full_prompts)
model_l.set_classes(full_prompts, text_pe)
pwvo_keys = list(PWVO.keys())

r = model_l.predict(img_path, conf=0.15, verbose=False)[0]

img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
H, W = img_rgb.shape[:2]

SUPER_COLORS_RGB = {
    'PET/Polyester':        (29,  158, 117),
    'HDPE/LDPE film':       (55,  138, 221),
    'Polypropylene':        (216, 90,  48),
    'Polystyrene/EPS':      (186, 117, 23),
    'Multilayer/composite': (212, 83,  126),
    'Other':                (136, 128, 128),
}

# ── Compute masks ─────────────────────────────────────────────────
combined_mask = np.zeros((H, W), dtype=np.uint8)
class_masks   = {}
total_pixels  = H * W

if r.masks is not None:
    pmasks = r.masks.data.cpu().numpy()
    pconfs = r.boxes.conf.cpu().numpy()
    pcls   = r.boxes.cls.cpu().numpy().astype(int)

    if pmasks.shape[1:] != (H, W):
        pmasks = np.stack([cv2.resize(pmasks[k], (W, H),
                           interpolation=cv2.INTER_NEAREST)
                           for k in range(len(pmasks))])

    for k, (pc, pci) in enumerate(zip(pconfs, pcls)):
        if pci >= len(pwvo_keys): continue
        super_c  = PWVO.get(pwvo_keys[pci], {}).get('super', 'Other')
        mask_bin = (pmasks[k] > 0.5).astype(np.uint8)
        combined_mask = np.maximum(combined_mask, mask_bin)
        if super_c not in class_masks:
            class_masks[super_c] = np.zeros((H, W), dtype=np.uint8)
        class_masks[super_c] = np.maximum(class_masks[super_c], mask_bin)

combined_pct = combined_mask.sum() / total_pixels * 100
class_pcts = {sc: m.sum() / total_pixels * 100
              for sc, m in class_masks.items()}

# ── Build figure ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.subplots_adjust(wspace=0.03, left=0.01, right=0.99,
                    top=0.88, bottom=0.04)

# Panel 1 — original
axes[0].imshow(img_rgb)
axes[0].set_title('Original frame', fontsize=10, fontweight='bold')
axes[0].set_xticks([]); axes[0].set_yticks([])

# Panel 2 — combined binary mask
axes[1].imshow(combined_mask, cmap='gray')
axes[1].set_title(f'Combined mask\n({combined_pct:.1f}% frame coverage)',
                  fontsize=10, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

# Panel 3 — per-class coloured overlay
axes[2].imshow(img_rgb)
color_overlay = np.zeros((H, W, 4))
for sc, m in class_masks.items():
    color = SUPER_COLORS_RGB.get(sc, (136, 128, 128))
    rgba  = [c/255 for c in color] + [0.55]
    color_overlay[m > 0] = rgba
axes[2].imshow(color_overlay)
axes[2].set_title('Per-class coverage breakdown',
                  fontsize=10, fontweight='bold')
axes[2].set_xticks([]); axes[2].set_yticks([])

# ── Inset bar chart — fixed ───────────────────────────────────────
sorted_classes = sorted(class_pcts.items(), key=lambda x: -x[1])
labels = [sc.split('/')[0] for sc, _ in sorted_classes]
vals   = [pct for _, pct in sorted_classes]
colors = [tuple(v/255 for v in SUPER_COLORS_RGB.get(sc, (136,128,128)))
          for sc, _ in sorted_classes]

ax_ins = axes[2].inset_axes([0.52, 0.05, 0.46, 0.45])
y_pos  = np.arange(len(labels))          # numeric positions — avoids category error
ax_ins.barh(y_pos, vals, color=colors, height=0.6)
ax_ins.set_yticks(y_pos)
ax_ins.set_yticklabels(labels, fontsize=7)
ax_ins.set_xlabel('Coverage (%)', fontsize=7)
ax_ins.tick_params(axis='x', labelsize=7)
ax_ins.set_facecolor('white')
for i, val in enumerate(vals):
    ax_ins.text(val + 0.1, y_pos[i], f'{val:.1f}%',
                va='center', fontsize=6.5)

fig.suptitle(
    'PlastiVision-Edge — Plastic Coverage Area Estimation\n'
    'YOLOE-26L + full PWVO vocabulary · conf ≥ 0.15 · '
    'zero additional inference cost',
    fontsize=10, fontweight='bold', y=0.98
)

out = RES_DIR / 'coverage_estimation.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n✅ Saved → {out}")
print(f"\nCoverage summary:")
print(f"  Combined: {combined_pct:.1f}%")
for sc, pct in sorted_classes:
    print(f"  {sc:<25} {pct:.1f}%")

In [ ]:
# ── Final figure — bar chart top-left ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.subplots_adjust(wspace=0.03, left=0.01, right=0.99,
                    top=0.95, bottom=0.04)

# Panel 1 — original
axes[0].imshow(img_rgb)
axes[0].set_title('(Left)  Original frame', fontsize=10, fontweight='bold')
axes[0].set_xticks([]); axes[0].set_yticks([])

# Panel 2 — combined binary mask
axes[1].imshow(combined_mask, cmap='gray')
axes[1].set_title(f'(Centre)  Combined binary mask\n'
                  f'11.0% frame coverage  ·  conf \u2265 0.30',
                  fontsize=10, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

# Panel 3 — per-class coloured overlay
axes[2].imshow(img_rgb)
color_overlay = np.zeros((H, W, 4))
for sc, m in class_masks.items():
    color = SUPER_COLORS_RGB.get(sc, (136, 128, 128))
    rgba  = [c/255 for c in color] + [0.55]
    color_overlay[m > 0] = rgba
axes[2].imshow(color_overlay)
axes[2].set_title('(Right)  Per-class coverage\n'
                  'YOLOE-26L · full PWVO · zero additional inference cost',
                  fontsize=10, fontweight='bold')
axes[2].set_xticks([]); axes[2].set_yticks([])

# Inset bar chart — top-left corner
sorted_classes = sorted(class_pcts.items(), key=lambda x: -x[1])
labels     = [sc.split('/')[0] for sc, _ in sorted_classes]
vals       = [pct for _, pct in sorted_classes]
colors_bar = [tuple(v/255 for v in SUPER_COLORS_RGB.get(sc, (136,128,128)))
              for sc, _ in sorted_classes]

ax_ins = axes[2].inset_axes([0.01, 0.58, 0.46, 0.38])  # top-left
y_pos  = np.arange(len(labels))
ax_ins.barh(y_pos, vals, color=colors_bar, height=0.5)
ax_ins.set_yticks(y_pos)
ax_ins.set_yticklabels(labels, fontsize=9)
ax_ins.set_xlabel('Coverage (%)', fontsize=8)
ax_ins.tick_params(axis='x', labelsize=8)
ax_ins.set_facecolor('white')
ax_ins.set_alpha(0.92)
ax_ins.spines['top'].set_visible(False)
ax_ins.spines['right'].set_visible(False)
for i, val in enumerate(vals):
    ax_ins.text(val + 0.1, y_pos[i], f'{val:.1f}%',
                va='center', fontsize=9, fontweight='bold')

out = RES_DIR / 'coverage_estimation.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✅ Saved → {out}")

In [ ]:
# ── Verify polymer discrimination on bubble wrap ──────────────────
bubble_img = str(FIG_DIR / 'bubble_wrap.jpg')

print("=== HDPE prompt on bubble wrap ===")
hdpe_pe = model_s.get_text_pe(['HDPE'])
model_s.set_classes(['HDPE'], hdpe_pe)
r_hdpe_s = model_s.predict(bubble_img, conf=0.05, verbose=False)[0]
conf_hdpe_s = float(r_hdpe_s.boxes.conf.max()) \
              if r_hdpe_s.boxes is not None and len(r_hdpe_s.boxes) > 0 \
              else 0.000
print(f"  YOLOE-26S: conf={conf_hdpe_s:.3f}")

hdpe_pe_l = model_l.get_text_pe(['HDPE'])
model_l.set_classes(['HDPE'], hdpe_pe_l)
r_hdpe_l = model_l.predict(bubble_img, conf=0.05, verbose=False)[0]
conf_hdpe_l = float(r_hdpe_l.boxes.conf.max()) \
              if r_hdpe_l.boxes is not None and len(r_hdpe_l.boxes) > 0 \
              else 0.000
print(f"  YOLOE-26L: conf={conf_hdpe_l:.3f}")

print("\n=== LDPE prompt on bubble wrap ===")
ldpe_pe = model_s.get_text_pe(['LDPE'])
model_s.set_classes(['LDPE'], ldpe_pe)
r_ldpe_s = model_s.predict(bubble_img, conf=0.05, verbose=False)[0]
conf_ldpe_s = float(r_ldpe_s.boxes.conf.max()) \
              if r_ldpe_s.boxes is not None and len(r_ldpe_s.boxes) > 0 \
              else 0.000
print(f"  YOLOE-26S: conf={conf_ldpe_s:.3f}")

ldpe_pe_l = model_l.get_text_pe(['LDPE'])
model_l.set_classes(['LDPE'], ldpe_pe_l)
r_ldpe_l = model_l.predict(bubble_img, conf=0.05, verbose=False)[0]
conf_ldpe_l = float(r_ldpe_l.boxes.conf.max()) \
              if r_ldpe_l.boxes is not None and len(r_ldpe_l.boxes) > 0 \
              else 0.000
print(f"  YOLOE-26L: conf={conf_ldpe_l:.3f}")

print(f"\n=== Summary ===")
print(f"  HDPE on bubble wrap: 26S={conf_hdpe_s:.3f}  26L={conf_hdpe_l:.3f}")
print(f"  LDPE on bubble wrap: 26S={conf_ldpe_s:.3f}  26L={conf_ldpe_l:.3f}")